In [ ]:
# ── CAPTURE CELL : Run this FIRST before any other cell ─────────────────────
# All print() output appears in BOTH the cell output AND ps1_outputs/console.log
# All plt.show() figures appear in BOTH the cell output AND ps1_outputs/<name>.png
import os, sys
OUT_DIR = "ps1_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class _Tee:
    """Writes to Jupyter cell output + log file simultaneously."""
    def __init__(self, jupyter_stream, log_fh):
        self._jupyter = jupyter_stream   # Jupyter OutStream -> shows in cell
        self._log     = log_fh           # file handle       -> saved to disk
        self._is_tee  = True
    def write(self, data):
        try:
            self._jupyter.write(data)
            self._jupyter.flush()
        except Exception:
            pass
        try:
            self._log.write(data)
            self._log.flush()
        except Exception:
            pass
    def flush(self):
        try: self._jupyter.flush()
        except Exception: pass
        try: self._log.flush()
        except Exception: pass
    def __getattr__(self, name):
        return getattr(self._jupyter, name)

if not getattr(sys.stdout, "_is_tee", False):
    _log_fh    = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _log_fh)
    sys.stderr = _Tee(sys.stderr, _log_fh)
    print("[capture] All output -> cell output + " + os.path.join(OUT_DIR, "console.log"))
else:
    print("[capture] Already active.")

try:
    import matplotlib.pyplot as _plt
    if not getattr(_plt.show, "_wrapped", False):
        _orig_show = _plt.show
        # Counter stored as mutable default arg on the function itself --
        # immune to any variable-name collision from other cells in the notebook.
        def _show(*a, _cnt=[0], **k):
            for _fig_num in _plt.get_fignums():
                _fig = _plt.figure(_fig_num)
                _title = (
                    (_fig._suptitle.get_text().strip() if _fig._suptitle else None)
                    or (_fig.axes[0].get_title().strip() if _fig.axes else None)
                    or f"fig_{_cnt[0]:02d}"
                )
                _safe = "".join(
                    c if c.isalnum() or c in "-_ " else "_" for c in _title
                ).strip().replace(" ", "_")[:60]
                _fname = os.path.join(OUT_DIR, f"{_safe}_{_cnt[0]:02d}.png")
                try:
                    _fig.savefig(_fname, dpi=110, bbox_inches="tight")
                    print(f"[capture] fig saved -> {_fname}")
                    _cnt[0] += 1
                except Exception as _save_err:
                    print(f"[capture] fig save failed: {_save_err}")
            return _orig_show(*a, **k)
        _show._wrapped = True
        _plt.show = _show
    print(f"[capture] matplotlib patched -> figures saved to {OUT_DIR}/")
except Exception as _cap_err:
    print("[capture] matplotlib not ready:", _cap_err)


**PS1 — Device Failure Prediction (3-day Binary) — TVM + GATE**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture : CUBIC MARS AWS v6
  Gold S3 (Delta) → Feature Store (offline+online) → Train 5 models + stacking
  → MLflow Experiment → MLflow Model Registry → SageMaker Endpoint
  → SageMaker Model Monitor (data quality + model quality)

Target  : will_fail_3d  (0=no failure within 3 days, 1=failure within 3 days)
Scope   : TVM + GATE only (VALIDATOR excluded: zero ServiceNow incidents in incident_root_cause)
Grain   : one row per DEVICE_ID per day
Run in  : SageMaker Studio JupyterLab  (kernel: Python 3 — Data Science 3.0)

---

In [ ]:
# ── CELL 1 : Install dependencies ─────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs", "pyarrow",
    "scikit-learn>=1.4",
    "xgboost>=2.0", "lightgbm>=4.3", "catboost>=1.2",
    "optuna>=3.6", "shap>=0.45",
    "mlflow>=2.13", "sagemaker-mlflow",
    "boto3", "sagemaker>=2.220,<3.0", "sagemaker-train",
    "pandas>=2.0", "matplotlib", "seaborn",
    "imbalanced-learn",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print("All packages installed.")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile

# deltalake removed: tables exported as Parquet (DeltaProtocolError on deletionVectors)
from sklearn.model_selection         import TimeSeriesSplit
from sklearn.preprocessing           import StandardScaler, LabelEncoder
from sklearn.pipeline                import Pipeline
from sklearn.linear_model            import LogisticRegression
from sklearn.ensemble                import (RandomForestClassifier,
                                             StackingClassifier)
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.metrics                 import (roc_auc_score, roc_curve,
                                             f1_score, precision_score, recall_score,
                                             average_precision_score, precision_recall_curve,
                                             confusion_matrix, classification_report)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation
from catboost  import CatBoostClassifier

import mlflow, mlflow.sklearn, mlflow.xgboost
from mlflow.models.signature         import infer_signature

try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 60)
print("Imports OK.")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────
# Fill in your artifact bucket name (the dedicated artefacts bucket)
ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX = "chicago/gold/device_ps1_daily"
GOLD_S3     = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION  = Session()
ROLE        = get_execution_role()
REGION      = boto3.Session().region_name
SM_CLIENT   = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT   = boto3.client("s3",        region_name=REGION)

# ── MLflow – SageMaker Managed Tracking Server ────────────────────────────
MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found – using local file fallback (EFS)")

mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)

# Per-category names (experiment / model / registry / endpoint) are built
# inside CELL 6b and overridden at the top of each CATEGORY loop.
GOLD_VERSION = 6

# ── Feature Store ─────────────────────────────────────────────────────────
# Online store = latest record per device for real-time inference.
# IAM role now has AmazonSageMakerFeatureStoreAccess on the artifacts bucket.
ENABLE_FEATURE_STORE = True
FS_GROUP_NAME        = "ps1-chicago-device-features-dev"

# ── Target horizon ────────────────────────────────────────────────────────
# "3d" → predict failure within 3 days (tighter signal, higher precision)
# "7d" → predict failure within 7 days (more positives, easier AUC signal)
TARGET_HORIZON = "3d"

print(f"Region               : {REGION}")
print(f"Gold S3              : {GOLD_S3}")
print(f"Artifacts            : s3://{ARTIFACT_BUCKET}/")
print(f"MLflow URI           : {MLFLOW_TRACKING_URI}")
print(f"Enable Feature Store : {ENABLE_FEATURE_STORE}  ({FS_GROUP_NAME})")
print(f"Target horizon       : will_fail_{TARGET_HORIZON}")


In [ ]:
# ── CELL 3b : Checkpoint — load saved state (skip Cells 4-6b if found) ───────
# Saves: df, FEATURE_COLS, TARGET, CATEGORIES (arrays + metadata per TVM/GATE)
# Path : /home/sagemaker-user/PS1/checkpoints/  (EFS — survives instance stop)
# Usage: Run Cells 1-3-3b → if CHECKPOINT_LOADED=True → jump to Cell 9 (Helpers)
#        Set FORCE_RELOAD=True to re-pull from S3 and rebuild features.

import os, joblib, pandas as _pd_ckpt

CKPT_DIR      = "/home/sagemaker-user/PS1/checkpoints"
FORCE_RELOAD  = False
CHECKPOINT_LOADED = False

_ckpt_df   = os.path.join(CKPT_DIR, "df.parquet")
_ckpt_meta = os.path.join(CKPT_DIR, "meta.joblib")

if not FORCE_RELOAD and os.path.exists(_ckpt_df) and os.path.exists(_ckpt_meta):
    print("[CHECKPOINT] Loading from", CKPT_DIR)
    df      = _pd_ckpt.read_parquet(_ckpt_df)
    _meta   = joblib.load(_ckpt_meta)
    FEATURE_COLS = _meta["FEATURE_COLS"]
    TARGET       = _meta["TARGET"]
    ID_COLS      = _meta["ID_COLS"]
    _cat_meta    = _meta["CATEGORIES_meta"]
    _cat_arrays  = joblib.load(os.path.join(CKPT_DIR, "categories_arrays.joblib"))

    # Rebuild CATEGORIES dict — DataFrames from df filter + arrays from file
    CATEGORIES = {}
    for cat in ["TVM", "GATE"]:
        if cat not in _cat_meta: continue
        cat_df = df[df["mars_device_category"] == cat].sort_values(
                    ["DEVICE_ID", "event_date"])
        tr = cat_df[cat_df["event_date"] <  "2025-10-01"]
        va = cat_df[(cat_df["event_date"] >= "2025-10-01") &
                    (cat_df["event_date"] <  "2026-01-01")]
        te = cat_df[cat_df["event_date"] >= "2026-01-01"]
        arrs = _cat_arrays[cat]
        CATEGORIES[cat] = {
            **_cat_meta[cat],
            "train_df": tr, "val_df": va, "test_df": te,
            "X_train": arrs["X_train"], "y_train": arrs["y_train"],
            "X_val":   arrs["X_val"],   "y_val":   arrs["y_val"],
            "X_test":  arrs["X_test"],  "y_test":  arrs["y_test"],
            "X_cv":    arrs["X_cv"],    "y_cv":    arrs["y_cv"],
        }
        print(f"  {cat}: train={len(tr):,}  val={len(va):,}  test={len(te):,}"
              f"  X_train={arrs['X_train'].shape}"
              f"  pos={CATEGORIES[cat]['pos']:,}")
        mlflow.set_experiment(CATEGORIES[cat]["experiment_name"])

    CHECKPOINT_LOADED = True
    print(f"\n  df={df.shape}  FEATURE_COLS={len(FEATURE_COLS)}  TARGET={TARGET}")
    print()
    print("  CHECKPOINT LOADED — skip Cells 4, 5, 6, 6b.")
    print("  Jump to Cell 7 (Feature Store) or Cell 9 (Helpers) if FS not needed.")
else:
    print("[CHECKPOINT] Not found or FORCE_RELOAD=True — running full pipeline (Cells 4-6b).")
    print("  Checkpoint will be saved automatically at end of Cell 6b.")


In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────
# G01 source: mars_dev.gold.device_ps1_daily exported via notebooks/export_gold_to_s3.py
# Column note: gold SQL writes "transit_day"; aliased to "event_date" below.

print(f"Loading Gold table from: {GOLD_S3}")

df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

# gold SQL writes transit_day; alias to event_date used throughout notebook
if "transit_day" in df.columns and "event_date" not in df.columns:
    df = df.rename(columns={"transit_day": "event_date"})
df["event_date"] = pd.to_datetime(df["event_date"])

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['event_date'].min().date()}  →  {df['event_date'].max().date()}")
print(f"\nTarget distribution:\n{df['will_fail_3d'].value_counts()}")
print(f"Failure rate    : {df['will_fail_3d'].mean():.2%}")
print(f"\nDevice mix:\n{df.drop_duplicates('DEVICE_ID')['mars_device_category'].value_counts()}")
# Derive will_fail_7d ? not in Parquet; computed as forward 7-day window
# of chargeable_outage_count > 0. Used when TARGET_HORIZON = '7d'.
if 'will_fail_7d' not in df.columns and 'chargeable_outage_count' in df.columns:
    df = df.sort_values(['DEVICE_ID', 'event_date']).reset_index(drop=True)
    def _fwd_max(x, w=7):
        rev = x.values[::-1]
        fwd = __import__('pandas').Series(rev).rolling(w, min_periods=1).max().shift(1).values[::-1]
        return __import__('pandas').Series(fwd, index=x.index)
    df['will_fail_7d'] = (
        df.groupby('DEVICE_ID')['chargeable_outage_count'].transform(_fwd_max) > 0
    ).astype(int).fillna(0).astype(int)
    print('Derived will_fail_7d:')
    for _cat in df['mars_device_category'].dropna().unique():
        _r = df[df['mars_device_category']==_cat]['will_fail_7d'].mean()
        print(f'  {_cat}: {_r:.2%} positive rate')
elif 'will_fail_7d' in df.columns:
    print(f"will_fail_7d already in Parquet: {df['will_fail_7d'].mean():.2%} positive rate")
else:
    print('WARNING: chargeable_outage_count not found ? will_fail_7d cannot be derived.')


In [ ]:
# ── CELL 5 : EDA ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df["will_fail_3d"].value_counts().plot(
    kind="bar", ax=axes[0], color=["#43A047","#E53935"], edgecolor="white")
axes[0].set_title("Target: will_fail_3d")

(1 - df.isnull().mean()).sort_values().head(20).plot(
    kind="barh", ax=axes[1], color="#1565C0")
axes[1].set_title("Feature completeness (lowest 20)")

df.groupby("event_date")["will_fail_3d"].sum().plot(
    ax=axes[2], color="#E53935", linewidth=0.8)
axes[2].set_title("Daily failure count over time")

plt.tight_layout()
plt.savefig("/tmp/ps1_eda.png", dpi=100)
plt.show()
print("EDA saved → /tmp/ps1_eda.png")

In [ ]:
# ── CELL 6 : Feature engineering (global — all categories) ──────────────────
TARGET   = f"will_fail_{TARGET_HORIZON}"

# R3-2: Exclude VALIDATOR
df = df[df["mars_device_category"].isin(["TVM", "GATE"])].copy()
print(f"After VALIDATOR exclusion: {df.shape}  ({df['mars_device_category'].value_counts().to_dict()})")

# Sort once — all time-windowed features below depend on this

# ?? Align Gold Parquet column names to notebook expectations ??????????????
# Gold Parquet uses transit_day / event_count / oos_event_count;
# feature blocks below expect event_date / daily_event_count / oos_flag.
if 'transit_day' in df.columns and 'event_date' not in df.columns:
    df = df.rename(columns={'transit_day': 'event_date'})
if 'event_count' in df.columns and 'daily_event_count' not in df.columns:
    df = df.rename(columns={'event_count': 'daily_event_count'})
if 'oos_event_count' in df.columns and 'oos_flag' not in df.columns:
    df['oos_flag'] = (df['oos_event_count'] > 0).astype(int)
    print(f'Derived oos_flag from oos_event_count (mean={df["oos_flag"].mean():.3f})')

df = df.sort_values(["DEVICE_ID", "event_date"]).reset_index(drop=True)

ID_COLS  = ["DEVICE_ID", "event_date", "split"]

# ── Cast decimal.Decimal columns to float64 ───────────────────────────────────
# Delta Lake / Databricks exports DECIMAL columns as Python decimal.Decimal objects.
# pandas cannot do arithmetic between decimal.Decimal and float — cast everything up front.
import decimal as _decimal
_decimal_cols = [
    col for col in df.columns
    if df[col].dtype == object
    and len(df[col].dropna()) > 0
    and isinstance(df[col].dropna().iloc[0], _decimal.Decimal)
]
if _decimal_cols:
    df[_decimal_cols] = df[_decimal_cols].apply(pd.to_numeric, errors='coerce')
    print(f"Cast {len(_decimal_cols)} decimal.Decimal column(s) to float64: {_decimal_cols}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK A — Rolling aggregates (original)
# ══════════════════════════════════════════════════════════════════════════════
for col, window in [("oos_flag", 7), ("oos_flag", 30), ("daily_event_count", 7)]:
    if col in df.columns:
        df[f"{col}_roll{window}"] = (
            df.groupby("DEVICE_ID")[col]
              .transform(lambda x: x.rolling(window, min_periods=1).sum()))

if "daily_event_count" in df.columns:
    df["event_rate_zscore"] = (
        df.groupby("DEVICE_ID")["daily_event_count"]
          .transform(lambda x: (x - x.mean()) / (x.std() + 1e-6)))

if "use_revenue_decline_flag" in df.columns:
    df["use_revenue_decline_flag"] = df["use_revenue_decline_flag"].astype(int)

if "use_revenue_cents" in df.columns:
    df["use_revenue_cents_roll7"] = (
        df.groupby("DEVICE_ID")["use_revenue_cents"]
          .transform(lambda x: x.rolling(7, min_periods=1).mean()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK B — Lag features t-1 / t-2 / t-3  (shift within device, no leakage)
# ══════════════════════════════════════════════════════════════════════════════
LAG_COLS = [c for c in ["oos_flag", "daily_event_count", "use_revenue_cents"] if c in df.columns]
for col in LAG_COLS:
    for lag in [1, 2, 3]:
        df[f"{col}_lag{lag}"] = df.groupby("DEVICE_ID")[col].shift(lag)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK C — Historical OOS counts + days-since + lifetime  (all shifted)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    for window in [30, 90]:
        df[f"oos_count_{window}d"] = (
            df.groupby("DEVICE_ID")["oos_flag"]
              .transform(lambda x, w=window: x.shift(1).rolling(w, min_periods=1).sum()))

    def _days_since_last_oos(x):
        cs = x.shift(1).cumsum()
        result = cs.groupby(cs).cumcount().astype(float)
        result[cs == 0] = np.nan
        return result

    df["days_since_last_oos"] = (
        df.groupby("DEVICE_ID")["oos_flag"].transform(_days_since_last_oos))

    df["oos_lifetime_count"] = (
        df.groupby("DEVICE_ID")["oos_flag"]
          .transform(lambda x: x.shift(1).cumsum()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK D — Device age
# ══════════════════════════════════════════════════════════════════════════════
df["device_age_days"] = df.groupby("DEVICE_ID")["event_date"].transform(
    lambda x: (x - x.min()).dt.days)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK E — Consecutive OOS streak  (acute deterioration signal)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    def _consec_oos(x):
        s = x.shift(1).fillna(0)
        grp = (s != s.shift(1)).cumsum()
        streak = s.groupby(grp).cumcount() + 1
        streak[s == 0] = 0.0
        return streak.astype(float)

    df["consec_oos_days"] = df.groupby("DEVICE_ID")["oos_flag"].transform(_consec_oos)
    df["max_streak_30d"]  = (
        df.groupby("DEVICE_ID")["consec_oos_days"]
          .transform(lambda x: x.shift(1).rolling(30, min_periods=1).max()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK F — OOS rate acceleration  (trend: worsening or recovering?)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    _oos_7d_recent = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).sum())
    _oos_7d_prev   = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(8).rolling(7, min_periods=1).sum())
    df["oos_rate_accel_7d"] = (_oos_7d_recent - _oos_7d_prev).fillna(0)

    _oos_30d_recent = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).sum())
    _oos_30d_prev   = df.groupby("DEVICE_ID")["oos_flag"].transform(
        lambda x: x.shift(31).rolling(30, min_periods=1).sum())
    df["oos_rate_accel_30d"] = (_oos_30d_recent - _oos_30d_prev).fillna(0)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK G — Exponentially weighted OOS  (recent OOS weighted higher than old)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    df["oos_ewm7"]  = (
        df.groupby("DEVICE_ID")["oos_flag"]
          .transform(lambda x: x.shift(1).ewm(span=7,  min_periods=1).mean()))
    df["oos_ewm30"] = (
        df.groupby("DEVICE_ID")["oos_flag"]
          .transform(lambda x: x.shift(1).ewm(span=30, min_periods=1).mean()))

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK H — Recovery count  (OOS→OK transitions = recurrence-prone devices)
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns:
    _oos_lag1 = df.groupby("DEVICE_ID")["oos_flag"].shift(1).fillna(0)
    _oos_lag2 = df.groupby("DEVICE_ID")["oos_flag"].shift(2).fillna(0)
    df["_recovery"] = ((_oos_lag2 == 1) & (_oos_lag1 == 0)).astype(float)
    df["recovery_count_30d"] = (
        df.groupby("DEVICE_ID")["_recovery"]
          .transform(lambda x: x.rolling(30, min_periods=1).sum()))
    df["recovery_count_90d"] = (
        df.groupby("DEVICE_ID")["_recovery"]
          .transform(lambda x: x.rolling(90, min_periods=1).sum()))
    df = df.drop(columns=["_recovery"])

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK I — Anomaly z-scores  (deviation from device's own rolling baseline)
# ══════════════════════════════════════════════════════════════════════════════
if "daily_event_count" in df.columns:
    _mu  = df.groupby("DEVICE_ID")["daily_event_count"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    _std = df.groupby("DEVICE_ID")["daily_event_count"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).std())
    df["event_count_zscore_30d"] = ((df["daily_event_count"] - _mu) / (_std + 1e-6)).fillna(0)

if "use_revenue_cents" in df.columns:
    _mu  = df.groupby("DEVICE_ID")["use_revenue_cents"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    _std = df.groupby("DEVICE_ID")["use_revenue_cents"].transform(
               lambda x: x.shift(1).rolling(30, min_periods=5).std())
    df["revenue_zscore_30d"] = ((df["use_revenue_cents"].astype(float) - _mu) / (_std + 1e-6)).fillna(0)

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK J — Calendar features
# ══════════════════════════════════════════════════════════════════════════════
df["day_of_week"]  = df["event_date"].dt.dayofweek
df["month"]        = df["event_date"].dt.month
df["day_of_month"] = df["event_date"].dt.day
df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)
df["quarter"]      = df["event_date"].dt.quarter

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK K — Peer comparison  (facility/location-level failure rate)
# ══════════════════════════════════════════════════════════════════════════════
_facility_col = next((c for c in df.columns
                      if any(k in c.lower() for k in ["facility", "station", "location", "site"])), None)
if _facility_col and "oos_flag" in df.columns:
    _peer_oos = (df.groupby(["event_date", _facility_col])["oos_flag"]
                   .transform("mean"))
    df["peer_oos_rate"] = _peer_oos
    df["oos_vs_peer"]   = df["oos_flag"] - _peer_oos

# ══════════════════════════════════════════════════════════════════════════════
# BLOCK L — Interaction features
# ══════════════════════════════════════════════════════════════════════════════
if "oos_flag" in df.columns and "daily_event_count" in df.columns:
    df["oos_x_event_count"] = (
        df.groupby("DEVICE_ID")["oos_flag"].shift(1).fillna(0)
        * df.groupby("DEVICE_ID")["daily_event_count"].shift(1).fillna(0))

if "oos_ewm7" in df.columns and "device_age_days" in df.columns:
    df["oos_ewm7_x_age"] = df["oos_ewm7"] * np.log1p(df["device_age_days"])

# ══════════════════════════════════════════════════════════════════════════════
# LABEL LEAK EXCLUSION + FEATURE_COLS
# ══════════════════════════════════════════════════════════════════════════════

# Cast pre-computed 7-day decimal columns to float64 so they enter FEATURE_COLS
# (pandas excludes decimal(N,M) from is_numeric_dtype)
for _dc in ['availability_pct_7d', 'outage_min_7d']:
    if _dc in df.columns:
        df[_dc] = pd.to_numeric(df[_dc], errors='coerce').astype(float)
        print(f'  Cast {_dc} to float64')

LABEL_LEAK_COLS = [
    "failure_level",
    "max_failure_level",     # renamed from failure_level in Gold schema
    "is_chargeable",
    "is_hardware_oos",
    "is_hardware_event",
    "duration_min",
    "oos_duration_min",
    "will_fail_7d",
    "will_fail_3d",   # exclude both horizons — active TARGET excluded separately
]

FEATURE_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET] + LABEL_LEAK_COLS
    and pd.api.types.is_numeric_dtype(df[c])
]

print(f"\n{'─'*60}")
print(f"Total features : {len(FEATURE_COLS)}")
blocks = {
    "A  rolling aggregates"  : [c for c in FEATURE_COLS if "_roll" in c or "zscore" == c.split("_")[-1]],
    "B  lag t-1/2/3"         : [c for c in FEATURE_COLS if "_lag" in c],
    "C  OOS history"         : [c for c in FEATURE_COLS if "oos_count" in c or "days_since" in c or "lifetime" in c],
    "D  device age"          : [c for c in FEATURE_COLS if "device_age" in c],
    "E  streak"              : [c for c in FEATURE_COLS if "streak" in c or "consec" in c],
    "F  acceleration"        : [c for c in FEATURE_COLS if "accel" in c],
    "G  EWM"                 : [c for c in FEATURE_COLS if "ewm" in c and "x_" not in c],
    "H  recovery"            : [c for c in FEATURE_COLS if "recovery" in c],
    "I  anomaly z-scores"    : [c for c in FEATURE_COLS if "zscore" in c],
    "J  calendar"            : [c for c in FEATURE_COLS if c in ["day_of_week","month","day_of_month","is_weekend","quarter"]],
    "K  peer"                : [c for c in FEATURE_COLS if "peer" in c or "vs_peer" in c],
    "L  interactions"        : [c for c in FEATURE_COLS if "_x_" in c],
}
for name, cols in blocks.items():
    if cols:
        print(f"  Block {name:<28}: {len(cols):2d} features")
print(f"{'─'*60}")
print(f"Leak cols excluded: {[c for c in LABEL_LEAK_COLS if c in df.columns]}")
print("Feature engineering complete — run CELL 6b → CELL 10 → CELLs 11-23")

In [ ]:
# ── CELL 6b : Per-category temporal splits & resource config ─────────────────
# TVM and GATE trained as SEPARATE models.
# Each gets its own: split, scale_pos_weight, MLflow experiment,
# Feature Store group, Model Registry, and SageMaker endpoint.

CATEGORIES = {}
for cat in ["TVM", "GATE"]:
    cat_df = df[df["mars_device_category"] == cat].sort_values(["DEVICE_ID", "event_date"])

    tr = cat_df[cat_df["event_date"] <  "2025-10-01"]
    va = cat_df[(cat_df["event_date"] >= "2025-10-01") & (cat_df["event_date"] < "2026-01-01")]
    te = cat_df[cat_df["event_date"] >= "2026-01-01"]

    n0, n1 = (tr[TARGET] == 0).sum(), (tr[TARGET] == 1).sum()

    CATEGORIES[cat] = {
        "train_df": tr, "val_df": va, "test_df": te,
        "X_train": tr[FEATURE_COLS].values, "y_train": tr[TARGET].values,
        "X_val":   va[FEATURE_COLS].values, "y_val":   va[TARGET].values,
        "X_test":  te[FEATURE_COLS].values, "y_test":  te[TARGET].values,
        "X_cv":    np.vstack([tr[FEATURE_COLS].values, va[FEATURE_COLS].values]),
        "y_cv":    np.concatenate([tr[TARGET].values, va[TARGET].values]),
        "scale_pos_weight": round(n0 / max(n1, 1), 2),
        "neg": n0, "pos": n1,
        "experiment_name": f"chicago-ps1-3d-{cat.lower()}-failure",
        "model_name":      f"ps1-3d-{cat.lower()}-device-failure",
        "registry_name":   f"chicago-ps1-3d-{cat.lower()}-failure",
        "fs_group":        f"chicago-ps1-3d-{cat.lower()}-features",
        "endpoint":        f"chicago-ps1-3d-{cat.lower()}-failure-v1",
        # Recall floor for threshold tuning on val PR curve.
        # GATE: only 0.60% positive rate (scale_pos_weight ~166) —
        # recall=0.80 may not be achievable; use 0.70 as the floor.
        "recall_floor": 0.80 if cat == "TVM" else 0.70,
    }
    mlflow.set_experiment(CATEGORIES[cat]["experiment_name"])

    print(f"\n── {cat} ──────────────────────────────────────────────────────")
    print(f"  Train  : {len(tr):>8,}  ({tr['event_date'].min().date()} -> {tr['event_date'].max().date()})")
    print(f"  Val    : {len(va):>8,}")
    print(f"  Test   : {len(te):>8,}")
    print(f"  Pos rate (train): {n1/max(len(tr),1):.2%}  ({n1:,} positives)")
    print(f"  scale_pos_weight: {CATEGORIES[cat]['scale_pos_weight']}")
    print(f"  MLflow experiment: {CATEGORIES[cat]['experiment_name']}")

# ── Save checkpoint after per-category splits ────────────────────────────
import os, joblib as _jl
_ckpt_dir = "/home/sagemaker-user/PS1/checkpoints"
os.makedirs(_ckpt_dir, exist_ok=True)
df.to_parquet(os.path.join(_ckpt_dir, "df.parquet"), index=True)
# Save scalar/string metadata per category (no DataFrames or arrays)
_scalar_keys = ["scale_pos_weight","neg","pos","recall_floor",
                "experiment_name","model_name","registry_name",
                "fs_group","endpoint"]
_jl.dump({
    "FEATURE_COLS":    FEATURE_COLS,
    "TARGET":          TARGET,
    "ID_COLS":         ID_COLS,
    "CATEGORIES_meta": {
        cat: {k: v for k, v in data.items() if k in _scalar_keys}
        for cat, data in CATEGORIES.items()
    },
}, os.path.join(_ckpt_dir, "meta.joblib"))
# Save numpy arrays per category separately (can be large)
_jl.dump({
    cat: {k: data[k] for k in ["X_train","y_train","X_val","y_val",
                                 "X_test","y_test","X_cv","y_cv"]}
    for cat, data in CATEGORIES.items()
}, os.path.join(_ckpt_dir, "categories_arrays.joblib"))
print(f"[CHECKPOINT] Saved to {_ckpt_dir}")
for cat, data in CATEGORIES.items():
    print(f"  {cat}: X_train={data['X_train'].shape}  pos={data['pos']:,}")
print(f"  Next kernel restart: run Cells 1-3-3b only, then jump to Cell 9.")


In [ ]:
# -- CELL 7 : Feature Store -- create group + seed ONLINE store ---------------
if not ENABLE_FEATURE_STORE:
    print('Feature Store disabled -- skipping CELL 7.')
else:
    from sagemaker.feature_store.feature_group import FeatureGroup
    from sagemaker.feature_store.feature_definition import FeatureDefinition, FeatureTypeEnum
    import time

    OFFLINE_URI = f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}"

    # Latest record per device (full df -- split not yet assigned in CELL 9)
    fs_cols = [c for c in FEATURE_COLS + [TARGET, 'DEVICE_ID', 'event_date'] if c in df.columns]
    fs_full = df[fs_cols].copy()
    fs_full['DEVICE_ID'] = fs_full['DEVICE_ID'].astype(str)
    fs_latest = (fs_full.sort_values('event_date')
                         .groupby('DEVICE_ID', as_index=False).tail(1)
                         .reset_index(drop=True))
    fs_latest['event_time'] = fs_latest['event_date'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    fs_latest = fs_latest.drop(columns=['event_date'])
    num_cols = fs_latest.select_dtypes(include='number').columns
    fs_latest[num_cols] = fs_latest[num_cols].fillna(0)
    print(f'Candidate columns for Feature Store: {len(fs_latest.columns)}')

    def fs_type(dt):
        if pd.api.types.is_float_dtype(dt):   return FeatureTypeEnum.FRACTIONAL
        if pd.api.types.is_integer_dtype(dt): return FeatureTypeEnum.INTEGRAL
        return FeatureTypeEnum.STRING

    def _make_fg(cols_df):
        return FeatureGroup(
            name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
            feature_definitions=[FeatureDefinition(c, fs_type(cols_df[c].dtype))
                                 for c in cols_df.columns])

    def _fg_status(fg):
        try: return fg.describe().get('FeatureGroupStatus')
        except Exception: return None

    def _fg_registered_cols(fg):
        try: return {f['FeatureName'] for f in fg.describe().get('FeatureDefinitions', [])}
        except Exception: return set()

    # ?? Delete stale Feature Group if schema changed ??????????????????????????
    _probe = _make_fg(fs_latest)
    st = _fg_status(_probe)
    if st in ('Created', 'CreateFailed'):
        existing = _fg_registered_cols(_probe)
        if st == 'CreateFailed' or (set(fs_latest.columns) - existing):
            print(f'Deleting stale Feature Group (status={st})...')
            _probe.delete()
            while True:
                try: _make_fg(fs_latest).describe(); print('  Still deleting...'); time.sleep(10)
                except Exception: print('  Deleted.'); break
            st = None
        else:
            print(f'Schema OK ({len(existing)} cols) -- reusing existing Feature Group.')

    # ?? Create fresh Feature Group ????????????????????????????????????????????
    if st is None:
        feature_group = _make_fg(fs_latest)
        feature_group.create(
            s3_uri=OFFLINE_URI, record_identifier_name='DEVICE_ID',
            event_time_feature_name='event_time', role_arn=ROLE,
            enable_online_store=True,
            description='PS1 daily device features -- Chicago Ventra')
        print('Creating Feature Group...')
        while _fg_status(feature_group) != 'Created':
            time.sleep(15); print('  status:', _fg_status(feature_group))
        print('Waiting 60s for schema propagation...')
        time.sleep(60)
    else:
        feature_group = _probe

    # ?? Verify what actually got registered (SDK may drop engineered cols) ???
    registered = _fg_registered_cols(feature_group)
    unregistered = set(fs_latest.columns) - registered
    if unregistered:
        print(f'NOTE: {len(unregistered)} columns not registered by SDK (likely in-memory only):')
        print(f'  {sorted(unregistered)}')
        print(f'Ingesting {len(registered)} registered columns only.')
        fs_ingest = fs_latest[[c for c in fs_latest.columns if c in registered]].copy()
    else:
        fs_ingest = fs_latest
        print(f'All {len(registered)} columns registered -- ingesting full set.')

    print(f'Feature Group ready: {FS_GROUP_NAME}  ({len(registered)} features)')

    # ?? Seed online store + verify ????????????????????????????????????????????
    feature_group.ingest(data_frame=fs_ingest.reset_index(drop=True), max_workers=4, wait=True)
    sample_id = fs_ingest['DEVICE_ID'].iloc[0]
    rec = feature_group.get_record(record_identifier_value_as_string=sample_id)
    print(f'Ingested {len(fs_ingest):,} records | online sample ({sample_id}):',
          {f['FeatureName']: f['ValueAsString'] for f in (rec or [])[:4]})
    globals()['feature_group'] = feature_group


In [ ]:
# ── CELL 8 : Feature Store — spot-check online store per category ────────────
# Creation and ingestion are in CELL 7. Verifies one TVM + one GATE device.
if not ENABLE_FEATURE_STORE:
    print("Feature Store disabled — skipping CELL 8.")
else:
    for cat_name in ["TVM", "GATE"]:
        sample_ids = (df[df["mars_device_category"] == cat_name]["DEVICE_ID"]
                      .astype(str).unique()[:1])
        if len(sample_ids):
            rec = feature_group.get_record(record_identifier_value_as_string=sample_ids[0])
            print(f"[{cat_name}] online store ({sample_ids[0]}):",
                  {f['FeatureName']: f['ValueAsString'] for f in (rec or [])[:4]})
        else:
            print(f"[{cat_name}] no devices found for spot-check")
    print("Feature Store verification complete.")


In [ ]:
# -- CELL 9 : Helper functions (evaluation -- train / val / test consistent) -----
# X_train/val/test, y_train/val/test are set at the top of each CATEGORY loop.

def make_common_tags(category):
    return {
        "ps"           : "PS1",
        "category"     : category,
        "target"       : TARGET,
        "city"         : "chicago",
        "gold_version" : str(GOLD_VERSION),
        "grain"        : "device_day",
        "architecture" : "CUBIC-MARS-AWS-v6",
    }


def evaluate_all(model, name):
    """Train / val / test evaluation in one call.

    Displays inline:
      1. Metrics summary table  (AUC / AP / Acc / F1 / Prec / Rec x train/val/test)
      2. Confusion matrices     (3-panel: train / val / test)
      3. ROC + Precision-Recall curves  (3 coloured lines each)
      4. Classification reports (all three splits, printed + logged)

    Logs all artefacts to the active MLflow run.
    Returns a flat dict of all split metrics for mlflow.log_metrics().
    """
    from sklearn.metrics import accuracy_score as _acc_fn
    splits = [('train', X_train, y_train),
               ('val',   X_val,   y_val),
               ('test',  X_test,  y_test)]
    all_m  = {}
    probs  = {}

    # ?? 1. scalar metrics (all three splits) ??????????????????????????????
    for split, X, y in splits:
        prob = model.predict_proba(X)[:, 1]
        pred = (prob >= 0.5).astype(int)
        probs[split] = prob
        all_m[f'{split}_auc']  = round(roc_auc_score(y, prob), 4)
        all_m[f'{split}_ap']   = round(average_precision_score(y, prob), 4)
        all_m[f'{split}_acc']  = round(_acc_fn(y, pred), 4)
        all_m[f'{split}_f1']   = round(f1_score(y, pred), 4)
        all_m[f'{split}_prec'] = round(precision_score(y, pred, zero_division=0), 4)
        all_m[f'{split}_rec']  = round(recall_score(y, pred, zero_division=0), 4)

    # ?? 2. metrics summary table ???????????????????????????????????????????
    _H = f'  {"Split":<6}  {"AUC":>7}  {"AP":>7}  {"Acc":>7}  {"F1":>7}  {"Prec":>7}  {"Rec":>7}'
    _S = f'  {"-"*6}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}'
    print(f'\n{name}')
    print(_H); print(_S)
    for _s, _, _ in splits:
        print(f'  {_s:<6}  '
              f'{all_m[_s+"_auc"]:>7.4f}  '
              f'{all_m[_s+"_ap"]:>7.4f}  '
              f'{all_m[_s+"_acc"]:>7.4f}  '
              f'{all_m[_s+"_f1"]:>7.4f}  '
              f'{all_m[_s+"_prec"]:>7.4f}  '
              f'{all_m[_s+"_rec"]:>7.4f}')

    # ?? 3. confusion matrices (3-panel) ???????????????????????????????????
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{name} -- Confusion Matrices', fontsize=12, fontweight='bold')
    for ax, (split, X, y) in zip(axes, splits):
        prob = probs[split]
        pred = (prob >= 0.5).astype(int)
        cm   = confusion_matrix(y, pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['No Fail', 'Fail'], yticklabels=['No Fail', 'Fail'])
        ax.set_title(
            f'{split.upper()}  AUC={all_m[split+"_auc"]:.3f}  Acc={all_m[split+"_acc"]:.3f}\n'
            f'F1={all_m[split+"_f1"]:.3f}  Prec={all_m[split+"_prec"]:.3f}  Rec={all_m[split+"_rec"]:.3f}')
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    _p = f'/tmp/cm_{name}.png'
    fig.savefig(_p, bbox_inches='tight', dpi=100); plt.show(); plt.close(fig)
    mlflow.log_artifact(_p)

    # ?? 4. ROC + Precision-Recall curves ??????????????????????????????????
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    _clr = {'train': '#1565C0', 'val': '#E65100', 'test': '#2E7D32'}
    for split, X, y in splits:
        prob        = probs[split]
        fpr, tpr, _ = roc_curve(y, prob)
        pr, rc, _   = precision_recall_curve(y, prob)
        col         = _clr[split]
        axes[0].plot(fpr, tpr, color=col, lw=1.6,
                     label=f'{split}  AUC={all_m[split+"_auc"]:.4f}')
        axes[1].plot(rc, pr, color=col, lw=1.6,
                     label=f'{split}  AP={all_m[split+"_ap"]:.4f}')
    axes[0].plot([0,1],[0,1], 'k--', lw=0.7)
    axes[0].set(xlabel='FPR', ylabel='TPR', title=f'{name} -- ROC Curve')
    axes[0].legend(fontsize=9)
    axes[1].set(xlabel='Recall', ylabel='Precision', title=f'{name} -- Precision-Recall')
    axes[1].legend(fontsize=9)
    plt.tight_layout()
    _p = f'/tmp/roc_pr_{name}.png'
    fig.savefig(_p, bbox_inches='tight', dpi=100); plt.show(); plt.close(fig)
    mlflow.log_artifact(_p)

    # ?? 5. classification reports (train / val / test) ?????????????????????
    print(f'\n{"="*66}')
    print(f'  {name} -- Classification Reports')
    print(f'{"="*66}')
    for split, X, y in splits:
        prob = probs[split]
        pred = (prob >= 0.5).astype(int)
        rpt  = classification_report(y, pred,
                                     target_names=['No Fail', 'Fail'], zero_division=0)
        print(f'\n  {split.upper()} set'
              f'  (AUC={all_m[split+"_auc"]:.4f}  Acc={all_m[split+"_acc"]:.4f}):')
        print(rpt)
        _p = f'/tmp/clf_rpt_{name}_{split}.txt'
        with open(_p, 'w') as _f:
            _f.write(f'{name} -- {split}\n{rpt}')
        mlflow.log_artifact(_p)

    return all_m


def log_feat_importance(importances, name, top_n=25):
    """Top-N feature importance bar chart -- shown inline and logged to MLflow."""
    if importances is None or len(importances) == 0:
        return
    n  = min(len(importances), len(FEATURE_COLS))
    fi = (pd.Series(importances[:n], index=FEATURE_COLS[:n])
            .sort_values(ascending=False).head(top_n))
    fig, ax = plt.subplots(figsize=(9, max(4, len(fi) * 0.35)))
    fi.plot.barh(ax=ax, color='steelblue', edgecolor='white')
    ax.invert_yaxis()
    ax.set_title(f'{name} -- Top {top_n} Feature Importances')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    _p = f'/tmp/feat_imp_{name}.png'
    fig.savefig(_p, bbox_inches='tight', dpi=100); plt.show(); plt.close(fig)
    mlflow.log_artifact(_p)
    _c = f'/tmp/feat_imp_{name}.csv'
    fi.reset_index().rename(columns={'index':'feature', 0:'importance'}).to_csv(_c, index=False)
    mlflow.log_artifact(_c)


In [ ]:
# ── CELL 9b : SMOTE checkpoint — load post-imputation/SMOTE arrays ───────────
# If checkpoint exists: restores imputed+SMOTEd arrays into CATEGORIES and
# initialises CATEGORY_RESULTS → skip Cell 10 entirely (saves 30-90 min).
# Set FORCE_SMOTE=True to re-run imputation + SMOTE from scratch.
# Requires Cell 3b checkpoint to have loaded first (CATEGORIES must exist).

import os, joblib as _jl_sm

_SMOTE_CKPT  = "/home/sagemaker-user/PS1/checkpoints/smote_arrays.joblib"
FORCE_SMOTE  = False
SMOTE_LOADED = False

if not FORCE_SMOTE and os.path.exists(_SMOTE_CKPT):
    print("[SMOTE CKPT] Loading post-imputation/SMOTE arrays ...")
    _smote_data = _jl_sm.load(_SMOTE_CKPT)
    CATEGORY_RESULTS = {cat: {} for cat in ["TVM", "GATE"]}
    for cat, arrs in _smote_data.items():
        CATEGORIES[cat]["X_train"]          = arrs["X_train"]
        CATEGORIES[cat]["y_train"]          = arrs["y_train"]
        CATEGORIES[cat]["X_val"]            = arrs["X_val"]
        CATEGORIES[cat]["X_test"]           = arrs["X_test"]
        CATEGORIES[cat]["X_cv"]             = arrs["X_cv"]
        CATEGORIES[cat]["pos"]              = arrs["pos"]
        CATEGORIES[cat]["scale_pos_weight"] = arrs["scale_pos_weight"]
        CATEGORY_RESULTS[cat]["train_median"] = arrs["train_median"]
        smote_tag = " [SMOTE applied]" if cat == "GATE" else ""
        print(f"  {cat}: X_train={arrs['X_train'].shape}"
              f"  pos={arrs['pos']:,} ({arrs['pos']/len(arrs['y_train']):.2%}){smote_tag}")
    SMOTE_LOADED = True
    print()
    print("  SMOTE CHECKPOINT LOADED — skip Cell 10.")
    print("  Jump to Cell 11 (MODEL 1 — Logistic Regression).")
else:
    print("[SMOTE CKPT] Not found or FORCE_SMOTE=True — run Cell 10 (imputation + SMOTE).")
    print("  Checkpoint will be saved automatically at end of Cell 10.")


In [ ]:
# ── CELL 10 : Per-category setup — imputation + SMOTE + CATEGORY_RESULTS ──────
# Run once before any training cell. CATEGORY_RESULTS accumulates across cells below.

# ?? Feature null-rate audit (train set, before imputation) ??????????????????
# Shows which FEATURE_COLS are effectively dead (>90% null) across each category.
# Dead columns survived the LABEL_LEAK_COLS filter but carry no real signal.
print('\n?? Feature null-rate audit (TRAIN set, before imputation) ??')
for _cat_name in ['TVM', 'GATE']:
    _tr = CATEGORIES[_cat_name]['train_df'][FEATURE_COLS]
    _nr = _tr.isnull().mean().sort_values(ascending=False)
    _dead   = _nr[_nr > 0.90]
    _sparse = _nr[(_nr > 0.10) & (_nr <= 0.90)]
    _full   = _nr[_nr <= 0.10]
    print(f'\n  {_cat_name}: {len(FEATURE_COLS)} features total')
    print(f'    Full   (<=10 % null): {len(_full):3d}')
    print(f'    Sparse (10-90% null): {len(_sparse):3d}')
    print(f'    Dead   (>90 % null): {len(_dead):3d}')
    if len(_dead):
        print(f'    Dead columns:')
        for _col in _dead.index:
            print(f'      {_col:45s}  {_dead[_col]:.1%} null')
    if len(_sparse):
        print(f'    Sparse columns (top 10):')
        for _col in _sparse.index[:10]:
            print(f'      {_col:45s}  {_sparse[_col]:.1%} null')
print()

CATEGORY_RESULTS = {cat: {} for cat in ['TVM', 'GATE']}

for CATEGORY in ['TVM', 'GATE']:
    cat = CATEGORIES[CATEGORY]

    # Cast to float64 before imputation (integer/object dtypes break nanmedian)
    for key in ('X_train', 'X_val', 'X_test', 'X_cv'):
        cat[key] = cat[key].astype(float)

    # Train-only median imputation — no leakage from val/test
    train_median = np.nanmedian(cat['X_train'], axis=0)
    cat['X_train'] = np.where(np.isnan(cat['X_train']), train_median, cat['X_train'])
    cat['X_val']   = np.where(np.isnan(cat['X_val']),   train_median, cat['X_val'])
    cat['X_test']  = np.where(np.isnan(cat['X_test']),  train_median, cat['X_test'])
    cat['X_cv']    = np.where(np.isnan(cat['X_cv']),    train_median, cat['X_cv'])
    CATEGORY_RESULTS[CATEGORY]['train_median'] = train_median

    # SMOTE oversampling for GATE only (0.6% positive → ~5% after resampling)
    # Applied to X_train/y_train only — val/test stay original for unbiased eval.
    if CATEGORY == 'GATE':
        from imblearn.over_sampling import SMOTE
        _sm = SMOTE(sampling_strategy=0.05, random_state=42)
        cat['X_train'], cat['y_train'] = _sm.fit_resample(cat['X_train'], cat['y_train'])
        cat['pos'] = int(cat['y_train'].sum())
        _n_smoted  = len(cat['y_train'])
        cat['scale_pos_weight'] = round((_n_smoted - cat['pos']) / cat['pos'], 2)

    n_total = len(cat['X_train'])
    print(f'\n── {CATEGORY} ──────────────────────────────────────────')
    print(f"  Train  : {n_total:,} rows  ({cat['pos']:,} pos = {cat['pos']/n_total:.2%})"
          + (" [SMOTE applied]" if CATEGORY == 'GATE' else ""))
    print(f"  Val    : {len(cat['X_val']):,}")
    print(f"  Test   : {len(cat['X_test']):,}")
    print(f"  scale_pos_weight : {cat['scale_pos_weight']}")
    print(f"  recall_floor     : {cat['recall_floor']}")
    print(f"  Experiment       : {cat['experiment_name']}")

print('\nSetup complete — run CELLs 11-23 in order.')

# ── Save post-imputation/SMOTE checkpoint ────────────────────────────────
import os, joblib as _jl_sm2
_smote_ckpt = "/home/sagemaker-user/PS1/checkpoints/smote_arrays.joblib"
_jl_sm2.dump({
    cat: {
        "X_train":          CATEGORIES[cat]["X_train"],
        "y_train":          CATEGORIES[cat]["y_train"],
        "X_val":            CATEGORIES[cat]["X_val"],
        "X_test":           CATEGORIES[cat]["X_test"],
        "X_cv":             CATEGORIES[cat]["X_cv"],
        "pos":              CATEGORIES[cat]["pos"],
        "scale_pos_weight": CATEGORIES[cat]["scale_pos_weight"],
        "train_median":     CATEGORY_RESULTS[cat]["train_median"],
    }
    for cat in ["TVM", "GATE"]
}, _smote_ckpt)
print(f"\n[SMOTE CKPT] Saved → {_smote_ckpt}")
for cat in ["TVM", "GATE"]:
    x = CATEGORIES[cat]["X_train"]
    print(f"  {cat}: X_train={x.shape}  pos={CATEGORIES[cat]['pos']:,}"
          f"  ({CATEGORIES[cat]['pos']/len(CATEGORIES[cat]['y_train']):.2%})")
print("  Next restart: load Cell 3b + Cell 9b → jump to Cell 11.")


In [ ]:
# ── CELL 11 : MODEL 1 — Logistic Regression (per category) ─────────────────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    with mlflow.start_run(run_name=f'lr_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'linear_baseline'})
        lr = Pipeline([('sc', StandardScaler()),
                        ('clf', LogisticRegression(C=0.1, class_weight='balanced',
                                                   max_iter=1000, solver='lbfgs', random_state=42))])
        lr.fit(X_train, y_train)
        mlflow.log_params({'C': 0.1, 'class_weight': 'balanced', 'scaler': 'standard'})
        all_m = evaluate_all(lr, f'lr_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(np.abs(lr.named_steps['clf'].coef_[0]), f'lr_{tag}')
        mlflow.sklearn.log_model(lr, 'model',
                                 signature=infer_signature(X_train, lr.predict_proba(X_train)),
                                 input_example=X_train[:3])
        CATEGORY_RESULTS[CATEGORY]['lr_run'] = run.info.run_id
    print(f'  [{CATEGORY}] LR   AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}')


In [ ]:
# ── CELL 12 : MODEL 2 — Random Forest (per category) ───────────────────────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    with mlflow.start_run(run_name=f'rf_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'bagging_ensemble'})
        rf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=20,
                                     class_weight='balanced_subsample', n_jobs=-1, random_state=42)
        rf.fit(X_train, y_train)
        mlflow.log_params({'n_estimators': 300, 'max_depth': 12, 'class_weight': 'balanced_subsample'})
        all_m = evaluate_all(rf, f'rf_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(rf.feature_importances_, f'rf_{tag}')
        mlflow.sklearn.log_model(rf, 'model',
                                 signature=infer_signature(X_train, rf.predict_proba(X_train)),
                                 input_example=X_train[:3])
        CATEGORY_RESULTS[CATEGORY]['rf_run'] = run.info.run_id
    print(f'  [{CATEGORY}] RF   AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}')


In [ ]:
# ── CELL 13 : MODEL 3 — XGBoost baseline (per category) ────────────────────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    XGB_PARAMS = dict(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, min_child_weight=10, gamma=1.0, reg_alpha=0.1,
                      reg_lambda=1.0, scale_pos_weight=SCALE_POS_WEIGHT, eval_metric='auc',
                      early_stopping_rounds=40, tree_method='hist', random_state=42, n_jobs=-1)
    with mlflow.start_run(run_name=f'xgb_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'gradient_boosting_xgb'})
        xgb = XGBClassifier(**XGB_PARAMS)
        xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
        mlflow.log_params({**XGB_PARAMS, 'best_iter': xgb.best_iteration})
        all_m = evaluate_all(xgb, f'xgb_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(xgb.feature_importances_, f'xgb_{tag}')
        sig = infer_signature(pd.DataFrame(X_train, columns=FEATURE_COLS), xgb.predict_proba(X_train))
        mlflow.xgboost.log_model(xgb, 'model', signature=sig,
                                  input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS))
        CATEGORY_RESULTS[CATEGORY]['xgb_run'] = run.info.run_id
    print(f'  [{CATEGORY}] XGB  AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}  best_iter={xgb.best_iteration}')


In [ ]:
# ── CELL 14 : MODEL 4 — LightGBM (per category) ────────────────────────────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    LGB_PARAMS = dict(n_estimators=600, num_leaves=63, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
                      scale_pos_weight=SCALE_POS_WEIGHT, n_jobs=-1, random_state=42, verbose=-1,
                      metric='auc')  # auc early-stopping prevents trivial all-zero convergence
    with mlflow.start_run(run_name=f'lgb_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'gradient_boosting_lgbm'})
        lgb = LGBMClassifier(**LGB_PARAMS)
        lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                callbacks=[early_stopping(40, verbose=False), log_evaluation(100)])
        mlflow.log_params({**LGB_PARAMS, 'best_iter': lgb.best_iteration_})
        all_m = evaluate_all(lgb, f'lgb_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(lgb.feature_importances_, f'lgb_{tag}')
        mlflow.sklearn.log_model(lgb, 'model',
                                 signature=infer_signature(X_train, lgb.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['lgb_run'] = run.info.run_id
    print(f'  [{CATEGORY}] LGB  AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}  best_iter={lgb.best_iteration_}')

In [ ]:
# ── CELL 15 : MODEL 5 — CatBoost (per category) ────────────────────────────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    cat_idx = [i for i, c in enumerate(FEATURE_COLS)
               if any(k in c for k in ('device_type', 'facility', 'subsystem', 'enc'))]
    CB_PARAMS = dict(iterations=600, depth=7, learning_rate=0.05, l2_leaf_reg=3,
                     scale_pos_weight=SCALE_POS_WEIGHT, eval_metric='AUC',
                     early_stopping_rounds=40, random_seed=42, task_type='CPU', verbose=100)
    with mlflow.start_run(run_name=f'cb_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'gradient_boosting_catboost'})
        cb = CatBoostClassifier(**CB_PARAMS)
        cb.fit(X_train, y_train, eval_set=(X_val, y_val),
               cat_features=cat_idx if cat_idx else None)
        mlflow.log_params({**{k: v for k, v in CB_PARAMS.items() if k != 'verbose'},
                           'best_iter': cb.best_iteration_, 'cat_features_count': len(cat_idx)})
        all_m = evaluate_all(cb, f'cb_{tag}')
        mlflow.log_metrics(all_m)
        log_feat_importance(cb.get_feature_importance(), f'cb_{tag}')
        cb.save_model(f'/tmp/ps1_{tag}_catboost.cbm')
        mlflow.log_artifact(f'/tmp/ps1_{tag}_catboost.cbm', 'catboost_native')
        mlflow.sklearn.log_model(cb, 'model',
                                 signature=infer_signature(X_train, cb.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['cb_run'] = run.info.run_id
    print(f'  [{CATEGORY}] CB   AUC={all_m["test_auc"]:.4f}  AP={all_m["test_ap"]:.4f}  Acc={all_m["test_acc"]:.4f}  F1={all_m["test_f1"]:.4f}  Rec={all_m["test_rec"]:.4f}  best_iter={cb.best_iteration_}')


In [ ]:
# ── CELL 16 : Optuna — XGBoost + LightGBM HP tuning (per category) ──────────
# SPEED FIX: best params saved to checkpoints/optuna_params.joblib after first
# run. Every subsequent run loads params and skips Optuna entirely (~0 sec).
# Set FORCE_OPTUNA=True to re-run the search (e.g. after major data changes).
# Trials: XGB=50 (was 150), LGB=40 (was 100). CV splits=3 (was 5). Timeout=20min.

import os as _os, joblib as _jl_opt

_OPT_CKPT    = "/home/sagemaker-user/PS1/checkpoints/optuna_params.joblib"
FORCE_OPTUNA = False
_RERUN_KEYS  = {"xgb_gate", "lgb_gate"}  # re-run both GATE models; TVM loads from checkpoint
_saved_params = {}
if not FORCE_OPTUNA and _os.path.exists(_OPT_CKPT):
    _saved_params = _jl_opt.load(_OPT_CKPT)
    print(f"[OPTUNA] Loaded saved params: {list(_saved_params.keys())}")
    print("  Set FORCE_OPTUNA=True to re-run search.")

for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()
    TS_CV = TimeSeriesSplit(n_splits=3)

    # ── XGBoost Optuna ───────────────────────────────────────────────────────
    _xgb_key = f"xgb_{tag}"
    _xgb_study = None  # only set if search runs (used for plots)
    if _xgb_key in _saved_params and not FORCE_OPTUNA and _xgb_key not in _RERUN_KEYS:
        BEST_PARAMS = {**_saved_params[_xgb_key]["best_params"],
                       "scale_pos_weight": SCALE_POS_WEIGHT,
                       "tree_method": "hist", "random_state": 42, "n_jobs": -1}
        _xgb_cv_ap = _saved_params[_xgb_key]["best_cv_ap"]
        print(f"\n  [{CATEGORY}] XGB: saved params loaded (CV_AP={_xgb_cv_ap:.4f}) — search skipped")
    else:
        print(f"\n  [{CATEGORY}] XGB: running Optuna (50 trials, timeout=20min) ...")
        def optuna_objective(trial):
            p = dict(
                n_estimators          = trial.suggest_int("n_estimators", 200, 800),
                max_depth             = trial.suggest_int("max_depth", 4, 9),
                learning_rate         = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                subsample             = trial.suggest_float("subsample", 0.5, 1.0),
                colsample_bytree      = trial.suggest_float("colsample_bytree", 0.5, 1.0),
                min_child_weight      = trial.suggest_int("min_child_weight", 1, 30),
                gamma                 = trial.suggest_float("gamma", 0.0, 3.0),
                reg_alpha             = trial.suggest_float("reg_alpha", 1e-4, 10., log=True),
                reg_lambda            = trial.suggest_float("reg_lambda", 1e-4, 10., log=True),
                scale_pos_weight      = SCALE_POS_WEIGHT,
                eval_metric           = "aucpr",
                early_stopping_rounds = 30,
                tree_method="hist", random_state=42, n_jobs=-1)
            scores = []
            for step, (tr, va) in enumerate(TS_CV.split(X_cv)):
                m = XGBClassifier(**p)
                m.fit(X_cv[tr], y_cv[tr], eval_set=[(X_cv[va], y_cv[va])], verbose=False)
                ap = average_precision_score(y_cv[va], m.predict_proba(X_cv[va])[:, 1])
                scores.append(ap)
                trial.report(np.mean(scores), step=step)
                if trial.should_prune(): raise optuna.TrialPruned()
            return np.mean(scores)

        _xgb_study = optuna.create_study(
            direction="maximize", study_name=f"ps1-xgb-{tag}",
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))
        optuna.logging.set_verbosity(optuna.logging.INFO)
        optuna.logging.set_verbosity(optuna.logging.INFO)
        _xgb_study.optimize(optuna_objective, n_trials=100, timeout=1800, show_progress_bar=False)
        _xgb_cv_ap = _xgb_study.best_value
        BEST_PARAMS = {**_xgb_study.best_params,
                       "scale_pos_weight": SCALE_POS_WEIGHT,
                       "tree_method": "hist", "random_state": 42, "n_jobs": -1}
        _saved_params[_xgb_key] = {"best_params": _xgb_study.best_params,
                                    "best_cv_ap": _xgb_cv_ap}
        _jl_opt.dump(_saved_params, _OPT_CKPT)
        print(f"  [{CATEGORY}] XGB params saved → {_OPT_CKPT}")

    CATEGORY_RESULTS[CATEGORY]["best_params"] = BEST_PARAMS

    with mlflow.start_run(run_name=f"xgb_optuna_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "xgboost_optuna_tuned"})
        mlflow.log_params({**BEST_PARAMS, "optuna_trials": 100, "optuna_cv_folds": 3,
                           "optuna_objective": "average_precision",
                           "optuna_best_cv_ap": round(_xgb_cv_ap, 4), "smote": False})
        xgb_tuned = XGBClassifier(**BEST_PARAMS)
        xgb_tuned.fit(X_cv, y_cv, verbose=False)
        all_m = evaluate_all(xgb_tuned, f"xgb_optuna_{tag}")
        mlflow.log_metrics({**all_m, "optuna_cv_ap": round(_xgb_cv_ap, 4)})
        log_feat_importance(xgb_tuned.feature_importances_, f"xgb_optuna_{tag}")
        if _xgb_study is not None:
            for fname, fig_fn in [
                (f"optuna_history_{tag}.png",      optuna.visualization.matplotlib.plot_optimization_history),
                (f"optuna_param_import_{tag}.png", optuna.visualization.matplotlib.plot_param_importances)]:
                try:
                    fig_fn(_xgb_study); plt.tight_layout()
                    plt.savefig(f"/tmp/{fname}", dpi=80); plt.close()
                    mlflow.log_artifact(f"/tmp/{fname}")
                except Exception: pass
        sig = infer_signature(pd.DataFrame(X_train, columns=FEATURE_COLS),
                              xgb_tuned.predict_proba(X_train))
        try:
            mlflow.xgboost.log_model(xgb_tuned, "model", signature=sig,
                                     input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS))
        except Exception as _lme: print(f"  [warn] log_model: {_lme}")
        CATEGORY_RESULTS[CATEGORY]["xgb_tuned_run"]   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]["xgb_tuned_model"] = xgb_tuned
    print(f"  [{CATEGORY}] XGB  CV_AP={_xgb_cv_ap:.4f}  AUC={all_m['test_auc']:.4f}  AP={all_m['test_ap']:.4f}  Rec={all_m['test_rec']:.4f}")

    # ── LightGBM Optuna ──────────────────────────────────────────────────────
    _lgb_key = f"lgb_{tag}"
    _lgb_study = None
    if _lgb_key in _saved_params and not FORCE_OPTUNA and _lgb_key not in _RERUN_KEYS:
        BEST_LGB_PARAMS = {**_saved_params[_lgb_key]["best_params"],
                           "scale_pos_weight": SCALE_POS_WEIGHT,
                           "metric": "binary_logloss",
                           "n_jobs": -1, "random_state": 42, "verbose": -1}
        _lgb_cv_ap = _saved_params[_lgb_key]["best_cv_ap"]
        print(f"\n  [{CATEGORY}] LGB: saved params loaded (CV_AP={_lgb_cv_ap:.4f}) — search skipped")
    else:
        print(f"\n  [{CATEGORY}] LGB: running Optuna (40 trials, timeout=20min) ...")
        def lgb_objective(trial):
            p = dict(
                n_estimators      = trial.suggest_int("n_estimators", 200, 800),
                max_depth         = trial.suggest_int("max_depth", 4, 10),
                learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                num_leaves        = trial.suggest_int("num_leaves", 20, 200),
                subsample         = trial.suggest_float("subsample", 0.5, 1.0),
                colsample_bytree  = trial.suggest_float("colsample_bytree", 0.5, 1.0),
                min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
                reg_alpha         = trial.suggest_float("reg_alpha", 1e-4, 10., log=True),
                reg_lambda        = trial.suggest_float("reg_lambda", 1e-4, 10., log=True),
                scale_pos_weight  = SCALE_POS_WEIGHT,
                n_jobs=-1, random_state=42, verbose=-1)  # no metric: n_estimators in search; early stopping removed
            scores = []
            for step, (tr, va) in enumerate(TS_CV.split(X_cv)):
                m = LGBMClassifier(**p)
                m.fit(X_cv[tr], y_cv[tr])  # n_estimators tuned by Optuna; no early stopping
                ap = average_precision_score(y_cv[va], m.predict_proba(X_cv[va])[:, 1])
                scores.append(ap)
                trial.report(np.mean(scores), step=step)
                if trial.should_prune(): raise optuna.TrialPruned()
            return np.mean(scores)

        _lgb_study = optuna.create_study(
            direction="maximize", study_name=f"ps1-lgb-{tag}",
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))
        optuna.logging.set_verbosity(optuna.logging.INFO)
        _lgb_study.optimize(lgb_objective, n_trials=60, timeout=1200, show_progress_bar=False)
        _lgb_cv_ap = _lgb_study.best_value
        BEST_LGB_PARAMS = {**_lgb_study.best_params,
                           "scale_pos_weight": SCALE_POS_WEIGHT,
                           "metric": "binary_logloss",
                           "n_jobs": -1, "random_state": 42, "verbose": -1}
        _saved_params[_lgb_key] = {"best_params": _lgb_study.best_params,
                                    "best_cv_ap": _lgb_cv_ap}
        _jl_opt.dump(_saved_params, _OPT_CKPT)
        print(f"  [{CATEGORY}] LGB params saved → {_OPT_CKPT}")

    CATEGORY_RESULTS[CATEGORY]["best_lgb_params"] = BEST_LGB_PARAMS

    with mlflow.start_run(run_name=f"lgb_optuna_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "lgbm_optuna_tuned"})
        mlflow.log_params({**BEST_LGB_PARAMS, "optuna_trials": 60, "optuna_cv_folds": 3,
                           "optuna_objective": "average_precision",
                           "optuna_best_cv_ap": round(_lgb_cv_ap, 4)})
        lgb_tuned = LGBMClassifier(**BEST_LGB_PARAMS)
        lgb_tuned.fit(X_cv, y_cv)
        all_m = evaluate_all(lgb_tuned, f"lgb_optuna_{tag}")
        mlflow.log_metrics({**all_m, "optuna_cv_ap": round(_lgb_cv_ap, 4)})
        log_feat_importance(lgb_tuned.feature_importances_, f"lgb_optuna_{tag}")
        sig = infer_signature(pd.DataFrame(X_train, columns=FEATURE_COLS),
                              lgb_tuned.predict_proba(X_train))
        try:
            mlflow.sklearn.log_model(lgb_tuned, "model", signature=sig,
                                     input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS),
                                     skops_trusted_types=["collections.OrderedDict",
                                                          "lightgbm.basic.Booster",
                                                          "lightgbm.sklearn.LGBMClassifier"])
        except Exception as _lme: print(f"  [warn] log_model: {_lme}")
        CATEGORY_RESULTS[CATEGORY]["lgb_tuned_run"]   = run.info.run_id
        CATEGORY_RESULTS[CATEGORY]["lgb_tuned_model"] = lgb_tuned
    print(f"  [{CATEGORY}] LGB  CV_AP={_lgb_cv_ap:.4f}  AUC={all_m['test_auc']:.4f}  AP={all_m['test_ap']:.4f}  Rec={all_m['test_rec']:.4f}")

print(f"\n[OPTUNA] Complete. Params at {_OPT_CKPT}")
print("  Next restart: params load automatically — Optuna skipped for all 4 keys.")
print("  Set FORCE_OPTUNA=True to re-run search.")


In [ ]:
# ── CELL 17 : Base model leaderboard (compare 5 models per category) ───────────
for CATEGORY in ['TVM', 'GATE']:
    cat = CATEGORIES[CATEGORY]
    mlflow.set_experiment(cat['experiment_name'])
    runs = mlflow.search_runs(
        experiment_names=[cat['experiment_name']],
        filter_string="tags.model_type != 'shap_analysis' AND tags.model_type != 'xgboost_multiclass'",
        order_by=['metrics.test_ap DESC'])
    print(f'\n── {CATEGORY} base model leaderboard (ranked by test_ap) ──')
    # Deduplicate: keep latest run per model name (handles notebook re-runs).
    if 'tags.mlflow.runName' in runs.columns and 'start_time' in runs.columns:
        runs = (runs.sort_values('start_time', ascending=False)
                    .drop_duplicates('tags.mlflow.runName')
                    .sort_values('metrics.test_ap', ascending=False, ignore_index=True))
    cols = ['tags.mlflow.runName', 'metrics.test_auc', 'metrics.test_ap',
            'metrics.test_f1', 'metrics.test_prec', 'metrics.test_rec',
            'metrics.train_auc', 'metrics.val_auc']
    available = [c for c in cols if c in runs.columns]
    print(runs[available].head(10).to_string(index=False))

In [ ]:
# ── CELL 18 : Stacking Ensemble (per category) ──────────────────────────────
# Base: LR + RF + XGB_tuned + LGB + CatBoost
# Final: CalibratedClassifierCV(LogisticRegression, isotonic)

# CatBoostClassifier mutates class_weights internally so sklearn clone() fails
# (get_params() doesn't return exactly what was passed to __init__).
# This thin BaseEstimator wrapper stores params as plain attrs; clone() works.
from sklearn.base import BaseEstimator, ClassifierMixin

class _CBWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, iterations=300, learning_rate=0.05,
                 class_weights=None, random_seed=42, verbose=0):
        self.iterations    = iterations
        self.learning_rate = learning_rate
        self.class_weights = class_weights
        self.random_seed   = random_seed
        self.verbose       = verbose

    def fit(self, X, y, **kw):
        self._cb = CatBoostClassifier(
            iterations=self.iterations, learning_rate=self.learning_rate,
            class_weights=self.class_weights, random_seed=self.random_seed,
            verbose=self.verbose)
        self._cb.fit(X, y, **kw)
        self.classes_ = np.array([0, 1])
        return self

    def predict(self, X):
        return self._cb.predict(X).flatten().astype(int)

    def predict_proba(self, X):
        return self._cb.predict_proba(X)


for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    BEST_PARAMS = CATEGORY_RESULTS[CATEGORY]['best_params']

    with mlflow.start_run(run_name=f'stack_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'stacking_ensemble'})
        mlflow.log_params({'base_models': 'LR+RF+XGB_tuned+LGB+CatBoost',
                           'final_estimator': 'LogisticRegression(balanced)+isotonic',
                           'cv_strategy': 'KFold(5)', 'passthrough': True})
        base = [
            ('lr',  Pipeline([('sc', StandardScaler()),
                              ('clf', LogisticRegression(C=0.1, class_weight='balanced',
                                                         max_iter=1000, random_state=42))])),
            ('rf',  RandomForestClassifier(n_estimators=200, max_depth=10,
                                           class_weight='balanced_subsample', n_jobs=-1, random_state=42)),
            ('xgb', XGBClassifier(**{**BEST_PARAMS, 'n_estimators': 300}, verbosity=0)),
            ('lgb', LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                   scale_pos_weight=SCALE_POS_WEIGHT, n_jobs=-1, random_state=42, verbose=-1)),
            ('cb',  _CBWrapper(iterations=300, learning_rate=0.05,
                               class_weights=[1, SCALE_POS_WEIGHT], random_seed=42, verbose=0)),
        ]
        stack = StackingClassifier(
            estimators=base,
            final_estimator=CalibratedClassifierCV(
                LogisticRegression(C=1.0, max_iter=500, class_weight='balanced'), cv=3, method='isotonic'),
            cv=5, stack_method='predict_proba',  # KFold(5): cross_val_predict requires full partitions; TimeSeriesSplit excludes early samples
            passthrough=True, n_jobs=-1)
        stack.fit(X_cv, y_cv)
        all_m = evaluate_all(stack, f'stack_{tag}')
        mlflow.log_metrics(all_m)
        mlflow.sklearn.log_model(stack, 'model',
                                 signature=infer_signature(X_train, stack.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CATEGORY_RESULTS[CATEGORY]['stack_run'] = run.info.run_id
    print(f'  [{CATEGORY}] Stack  AUC={all_m["test_auc"]}  AP={all_m["test_ap"]}  F1={all_m["test_f1"]}')


In [ ]:
# ── CELL 19 : SHAP — feature attribution (per category, on Optuna XGB) ─────────
# Runs for both TVM and GATE.
# Plots (per category): bar, beeswarm, scatter (top feature), waterfall (top failure)
# Printed analysis: top-20 importance table, waterfall drivers, missed-failure SHAP,
#                   TVM vs GATE comparison table.
_shap_imp_all = {}   # {CATEGORY: pd.Series(mean_abs_shap, index=features)}

for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT    = cat["scale_pos_weight"]
    EXPERIMENT_NAME     = cat["experiment_name"]
    MLFLOW_MODEL_NAME   = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR        = cat["recall_floor"]
    COMMON_TAGS = {**make_common_tags(CATEGORY),
                   "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    xgb_tuned = CATEGORY_RESULTS[CATEGORY]['xgb_tuned_model']
    _n_shap   = min(2000, len(X_test))
    X_shap    = pd.DataFrame(X_test[:_n_shap], columns=FEATURE_COLS)
    y_shap    = np.array(y_test[:_n_shap]).astype(int)

    # XGBoost 2.x stores base_score as '[value]' -- shap.TreeExplainer cannot parse it
    # (both Booster and XGBClassifier paths call save_config() internally).
    # Use XGBoost native pred_contribs=True: identical SHAP values, no SHAP loader.
    import xgboost as _xgb19
    _booster19  = xgb_tuned.get_booster()
    _dmat19     = _xgb19.DMatrix(X_shap.values, feature_names=list(X_shap.columns))
    _contribs19 = _booster19.predict(_dmat19, pred_contribs=True)
    # pred_contribs: (n_samples, n_features + 1) -- last col = bias term
    shap_values = shap.Explanation(
        values        = _contribs19[:, :-1],
        base_values   = _contribs19[:, -1],
        data          = X_shap.values,
        feature_names = list(X_shap.columns),
    )

    # Predicted probabilities (for failure-case selection and FN/TP split)
    _probs19 = xgb_tuned.predict_proba(X_shap)[:, 1]
    _pred19  = (_probs19 >= 0.5).astype(int)

    # Mean |SHAP| importance
    _mean_abs = pd.Series(np.abs(shap_values.values).mean(0), index=FEATURE_COLS)
    _shap_imp_all[CATEGORY] = _mean_abs.sort_values(ascending=False)
    top_feat = _shap_imp_all[CATEGORY].index[0]

    # Index of highest-confidence true positive (for waterfall)
    _tp_mask = (y_shap == 1) & (_pred19 == 1)
    if _tp_mask.any():
        _tp_cands = np.where(_tp_mask)[0]
        _tp_idx   = int(_tp_cands[np.argmax(_probs19[_tp_cands])])
    else:
        _tp_idx = int(np.argmax(_probs19))

    with mlflow.start_run(run_name=f'xgb_shap_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'shap_analysis',
                         'parent_run': CATEGORY_RESULTS[CATEGORY]['xgb_tuned_run']})
        # Bar + beeswarm (top 20 features)
        for sname, sfn in [('bar', shap.plots.bar), ('beeswarm', shap.plots.beeswarm)]:
            sfn(shap_values, max_display=20, show=False)
            fig = plt.gcf(); fig.tight_layout()
            fpath = f'/tmp/shap_{sname}_{tag}.png'
            fig.savefig(fpath, dpi=100); plt.close(fig)
            mlflow.log_artifact(fpath)
        # Scatter / dependency plot for top feature
        shap.plots.scatter(shap_values[:, top_feat], color=shap_values, show=False)
        fig = plt.gcf(); fig.tight_layout()
        fpath = f'/tmp/shap_dep_{tag}.png'
        fig.savefig(fpath, dpi=100); plt.close(fig)
        mlflow.log_artifact(fpath)
        # Waterfall for highest-confidence caught failure
        shap.plots.waterfall(shap_values[_tp_idx], max_display=15, show=False)
        fig = plt.gcf(); fig.tight_layout()
        fpath = f'/tmp/shap_waterfall_{tag}.png'
        fig.savefig(fpath, dpi=100); plt.close(fig)
        mlflow.log_artifact(fpath)
        # Importance CSV
        _shap_df = (_shap_imp_all[CATEGORY].reset_index()
                        .rename(columns={"index": "feature", 0: "mean_abs_shap"}))
        _shap_df.to_csv(f'/tmp/shap_importance_{tag}.csv', index=False)
        mlflow.log_artifact(f'/tmp/shap_importance_{tag}.csv')
        mlflow.log_param('top_shap_feature', top_feat)

    _SEP19 = "=" * 65
    # ── 1. Top-20 SHAP importance table ──────────────────────────────────────
    print(f"\n{_SEP19}")
    print(f"SHAP Feature Importance -- {CATEGORY}  (test n={_n_shap})")
    print(f"  mean |SHAP|: avg absolute impact on model output across all test samples")
    print(_SEP19)
    _top20 = _shap_imp_all[CATEGORY].head(20).reset_index()
    _top20.columns = ["feature", "mean_abs_shap"]
    _top20.insert(0, "rank", range(1, len(_top20) + 1))
    _top20["pct_total"] = (_top20["mean_abs_shap"] / _top20["mean_abs_shap"].sum() * 100).round(1)
    print(_top20.round(5).to_string(index=False))
    print(f"\n  Top feature : {top_feat}  (mean|SHAP|={_shap_imp_all[CATEGORY].iloc[0]:.5f})")

    # ── 2. Waterfall drivers (printed, top 10 for the sample failure) ────────
    _wf_sv  = pd.Series(shap_values.values[_tp_idx], index=FEATURE_COLS)
    _wf_top = _wf_sv.abs().sort_values(ascending=False).head(10)
    print(f"\n  Waterfall sample [{CATEGORY}] -- idx={_tp_idx}  "
          f"pred_prob={_probs19[_tp_idx]:.3f}  true_label={y_shap[_tp_idx]}")
    print(f"  {'Feature':<40}  {'SHAP':>9}  {'Raw value':>10}")
    print(f"  {'-'*40}  {'-'*9}  {'-'*10}")
    for feat in _wf_top.index:
        sv    = _wf_sv[feat]
        raw_v = X_shap.iloc[_tp_idx][feat]
        arrow = "+" if sv > 0 else "-"
        print(f"  {arrow} {feat:<39}  {sv:+9.5f}  {raw_v:10.3g}")

    # ── 3. Missed-failure analysis ────────────────────────────────────────────
    _fn_mask  = (y_shap == 1) & (_pred19 == 0)   # false negatives: missed failures
    _tp_mask2 = (y_shap == 1) & (_pred19 == 1)   # true positives:  caught failures
    n_tp = int(_tp_mask2.sum()); n_fn = int(_fn_mask.sum())
    print(f"\n{_SEP19}")
    print(f"Missed-Failure SHAP Analysis -- {CATEGORY}")
    print(f"  Caught (TP): {n_tp}  |  Missed (FN): {n_fn}")
    print(f"  Negative SHAP in FN = features pulling the score DOWN on missed alarms.")
    print(f"  Suppression = feature carries less SHAP weight in missed vs caught failures.")
    print(_SEP19)
    if n_fn >= 5 and n_tp >= 5:
        _sv_tp  = shap_values.values[_tp_mask2]   # (n_tp, n_feat)
        _sv_fn  = shap_values.values[_fn_mask]    # (n_fn, n_feat)
        _cmp19  = pd.DataFrame({
            "signed_TP":    _sv_tp.mean(0),
            "signed_FN":    _sv_fn.mean(0),
            "abs_TP":       np.abs(_sv_tp).mean(0),
            "abs_FN":       np.abs(_sv_fn).mean(0),
        }, index=FEATURE_COLS)
        _cmp19["suppression"] = _cmp19["abs_FN"] - _cmp19["abs_TP"]
        # Most suppressed: features that matter LESS for missed failures
        print("\n  Top 10 features suppressed in missed failures (carry less |SHAP| vs caught):")
        _supp = (_cmp19.sort_values("suppression").head(10)
                       [["abs_TP", "abs_FN", "signed_TP", "signed_FN", "suppression"]])
        print(_supp.round(5).to_string())
        # Features actively pushing score DOWN (most negative mean SHAP) in FN
        _neg_fn = _cmp19["signed_FN"].sort_values().head(10)
        print(f"\n  Top 10 features pushing score DOWN in missed failures (negative SHAP):")
        print(f"  {'Feature':<40}  {'FN SHAP':>9}  {'TP SHAP':>9}")
        print(f"  {'-'*40}  {'-'*9}  {'-'*9}")
        for feat, fn_v in _neg_fn.items():
            tp_v = _cmp19.loc[feat, "signed_TP"]
            print(f"  {feat:<40}  {fn_v:+9.5f}  {tp_v:+9.5f}")
    else:
        print(f"  [SKIP] Too few samples (FN={n_fn}, TP={n_tp}) -- need >= 5 each")

# ── 4. TVM vs GATE comparison table ──────────────────────────────────────────
if len(_shap_imp_all) == 2:
    _SEP19b = "=" * 70
    print(f"\n{_SEP19b}")
    print("TVM vs GATE -- SHAP Feature Importance Comparison")
    print("  Union of top-15 features from each model, sorted by TVM rank.")
    print("  rank_delta = |TVM_rank - GATE_rank|: large = important to one but not the other.")
    print(_SEP19b)
    _tvm_s  = _shap_imp_all["TVM"].rename("TVM")
    _gate_s = _shap_imp_all["GATE"].rename("GATE")
    _cmp2   = pd.concat([_tvm_s, _gate_s], axis=1).fillna(0.0)
    _cmp2["TVM_rank"]   = _cmp2["TVM"].rank(ascending=False, method="min").astype(int)
    _cmp2["GATE_rank"]  = _cmp2["GATE"].rank(ascending=False, method="min").astype(int)
    _cmp2["rank_delta"] = (_cmp2["TVM_rank"] - _cmp2["GATE_rank"]).abs()
    _union15 = (set(_shap_imp_all["TVM"].head(15).index)
              | set(_shap_imp_all["GATE"].head(15).index))
    _show2 = (_cmp2.loc[list(_union15)]
                   .sort_values("TVM_rank")
                   [["TVM_rank", "TVM", "GATE_rank", "GATE", "rank_delta"]])
    _show2.columns = ["TVM_rank", "TVM_|SHAP|", "GATE_rank", "GATE_|SHAP|", "rank_delta"]
    print(_show2.round(5).to_string())

    _tvm_top5  = list(_shap_imp_all["TVM"].head(5).index)
    _gate_top5 = list(_shap_imp_all["GATE"].head(5).index)
    _shared10  = sorted(set(_shap_imp_all["TVM"].head(10).index)
                      & set(_shap_imp_all["GATE"].head(10).index))
    _tvm_only  = [f for f in _tvm_top5  if f not in _shap_imp_all["GATE"].head(10).index]
    _gate_only = [f for f in _gate_top5 if f not in _shap_imp_all["TVM"].head(10).index]
    print(f"\n  TVM  top-5 : {_tvm_top5}")
    print(f"  GATE top-5 : {_gate_top5}")
    print(f"  Shared top-10 : {_shared10}")
    if _tvm_only:  print(f"  TVM-only  (top-5, not in GATE top-10) : {_tvm_only}")
    if _gate_only: print(f"  GATE-only (top-5, not in TVM  top-10) : {_gate_only}")


In [ ]:
# ── CELL 20 : Stage 2 — severity multiclass (per category, guarded) ────────────
# Skipped for GATE: failure_levels 2/3/4 have ~6/7/6 rows — min class < 20.
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    if 'failure_level' not in train_df.columns:
        print(f'  [{CATEGORY}] failure_level column absent — skip Stage 2'); continue

    sev_tr = train_df[train_df['will_fail_3d'] == 1].dropna(subset=['failure_level'])
    sev_te = test_df[test_df['will_fail_3d'] == 1].dropna(subset=['failure_level'])
    _class_counts = sev_tr['failure_level'].value_counts() if len(sev_tr) > 0 else pd.Series([])
    _skip = len(sev_tr) == 0 or _class_counts.min() < 20
    if _skip:
        print(f'  [{CATEGORY}] Stage 2 skipped — min class count'
              f" {_class_counts.min() if len(_class_counts) else 0} < 20")
        continue

    Xs_tr = sev_tr[FEATURE_COLS].values; ys_tr = sev_tr['failure_level'].values.astype(int) - 1
    Xs_te = sev_te[FEATURE_COLS].values; ys_te = sev_te['failure_level'].values.astype(int) - 1
    with mlflow.start_run(run_name=f'stage2_severity_{tag}') as run:
        mlflow.set_tags({**COMMON_TAGS, 'model_type': 'xgboost_multiclass',
                         'stage': '2', 'num_classes': '3'})
        sev_model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                                   objective='multi:softprob', num_class=3,
                                   tree_method='hist', random_state=42, n_jobs=-1)
        sev_model.fit(Xs_tr, ys_tr, verbose=False)
        sev_auc = roc_auc_score(ys_te, sev_model.predict_proba(Xs_te),
                                multi_class='ovr', average='macro')
        sev_f1 = f1_score(ys_te, sev_model.predict(Xs_te), average='macro')
        mlflow.log_metrics({'test_severity_auc': round(sev_auc, 4),
                            'test_severity_f1': round(sev_f1, 4)})
        CATEGORY_RESULTS[CATEGORY]['severity_model'] = sev_model
    print(f'  [{CATEGORY}] Stage2  AUC={sev_auc:.4f}  F1={sev_f1:.4f}')


In [ ]:
# ── CELL 21 : Threshold tuning + Champion selection (per category) ──────────────
# Threshold: tune on val PR curve, target recall >= recall_floor.
# Champion: ranked by test_ap (Average Precision) — right metric under heavy imbalance.
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    # ── Threshold tuning ─────────────────────────────────────────────────────────
    xgb_tuned = CATEGORY_RESULTS[CATEGORY]['xgb_tuned_model']
    y_prob_val_thresh = xgb_tuned.predict_proba(X_val)[:, 1]
    prec_curve, rec_curve, thresh_curve = precision_recall_curve(y_val, y_prob_val_thresh)
    candidates = [(t, p, r) for p, r, t in zip(prec_curve[:-1], rec_curve[:-1], thresh_curve)
                  if r >= RECALL_FLOOR]
    if candidates:
        BEST_THRESHOLD, BEST_PREC, BEST_REC = max(candidates, key=lambda x: x[1])
    else:
        BEST_THRESHOLD, BEST_PREC, BEST_REC = 0.5, float('nan'), float('nan')
        print(f'  [{CATEGORY}] No threshold achieves recall>={RECALL_FLOOR} — using 0.5')
    print(f'  [{CATEGORY}] Threshold: {BEST_THRESHOLD:.3f}'
          f'  prec={BEST_PREC:.3f}  rec={BEST_REC:.3f}  (recall>={RECALL_FLOOR} on val)')

    # ── Champion selection ────────────────────────────────────────────────────────
    all_runs = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        filter_string="tags.model_type != 'shap_analysis' AND tags.model_type != 'xgboost_multiclass'",
        order_by=['metrics.test_ap DESC'])
    CHAMPION_RUN_ID = all_runs.iloc[0]['run_id']
    CHAMPION_NAME   = all_runs.iloc[0]['tags.mlflow.runName']
    CHAMPION_AUC    = all_runs.iloc[0]['metrics.test_auc']
    CHAMPION_AP     = all_runs.iloc[0]['metrics.test_ap']
    print(f'  [{CATEGORY}] Champion: {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}  AP={CHAMPION_AP:.4f}')

    CATEGORY_RESULTS[CATEGORY].update({
        'champion_run_id': CHAMPION_RUN_ID, 'champion_name': CHAMPION_NAME,
        'champion_auc': CHAMPION_AUC, 'champion_ap': CHAMPION_AP,
        'threshold': BEST_THRESHOLD, 'threshold_prec': BEST_PREC, 'threshold_rec': BEST_REC,
    })

In [ ]:
# ── CELL 22 : Register champion in MLflow Model Registry (per category) ─────────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    CHAMPION_RUN_ID = CATEGORY_RESULTS[CATEGORY]['champion_run_id']
    CHAMPION_NAME   = CATEGORY_RESULTS[CATEGORY]['champion_name']
    CHAMPION_AP     = CATEGORY_RESULTS[CATEGORY]['champion_ap']
    CHAMPION_AUC    = CATEGORY_RESULTS[CATEGORY]['champion_auc']


    # ?? Gate-check: block registration unless champion meets quality floor ???????
    # Hard gate  : recall must meet RECALL_FLOOR (operational SLA ? missing failures is costly).
    # Soft gate  : AP must beat naive classifier (pos_rate = baseline for random predictions).
    # Both gates fail expected until Gold enrichment (maintenance/wear features) lands.
    champion_rec = CATEGORY_RESULTS[CATEGORY].get('threshold_rec', float('nan'))
    pos_rate     = cat['pos'] / max(cat['pos'] + cat['neg'], 1)   # naive-classifier baseline AP
    gate_issues  = []
    if not (champion_rec != champion_rec) and champion_rec < RECALL_FLOOR:   # nan-safe check
        gate_issues.append(f'recall={champion_rec:.3f} < floor={RECALL_FLOOR:.2f}')
    if CHAMPION_AP <= pos_rate:
        gate_issues.append(f'AP={CHAMPION_AP:.4f} <= baseline={pos_rate:.4f} (no better than random)')
    if gate_issues:
        print(f'\n  [{CATEGORY}] REGISTRATION BLOCKED ? gate(s) failed:')
        for _g in gate_issues:
            print(f'    * {_g}')
        print(f'  Note: expected until Gold enrichment (maintenance/wear/peer features) lands.')
        print(f'  Skipping {CATEGORY} MLflow registration.')
        continue
    print(f'  [{CATEGORY}] Gates passed ? recall={champion_rec:.3f}  '
          f'AP={CHAMPION_AP:.4f}  baseline={pos_rate:.4f}')

    mlfc = mlflow.tracking.MlflowClient()
    try:
        mlfc.create_registered_model(
            name=MLFLOW_MODEL_NAME,
            description=f'PS1-3d {CATEGORY} Device Failure — Chicago Ventra. Binary: will_fail_3d. v6.',
            tags={'ps': 'PS1_3d', 'category': CATEGORY, 'target': 'will_fail_3d', 'city': 'chicago'})
    except mlflow.exceptions.MlflowException: pass
    mv = mlfc.create_model_version(
        name=MLFLOW_MODEL_NAME, source=f'runs:/{CHAMPION_RUN_ID}/model',
        run_id=CHAMPION_RUN_ID,
        description=f'Champion: {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}  AP={CHAMPION_AP:.4f}',
        tags={'algorithm': CHAMPION_NAME, 'test_ap': str(round(CHAMPION_AP, 4)),
              'test_auc': str(round(CHAMPION_AUC, 4)), 'n_features': str(len(FEATURE_COLS))})
    mlfc.transition_model_version_stage(name=MLFLOW_MODEL_NAME, version=mv.version, stage='Staging')
    CATEGORY_RESULTS[CATEGORY]['mlflow_version'] = mv.version
    print(f'  [{CATEGORY}] MLflow: {MLFLOW_MODEL_NAME} v{mv.version} → Staging')


In [ ]:
# ── CELL 23 : Register champion in SageMaker Model Registry (per category) ──────
for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    RECALL_FLOOR      = cat["recall_floor"]
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    import tarfile, sagemaker.image_uris as sm_images
    CHAMPION_RUN_ID  = CATEGORY_RESULTS[CATEGORY]['champion_run_id']
    CHAMPION_NAME    = CATEGORY_RESULTS[CATEGORY]['champion_name']
    CHAMPION_AP      = CATEGORY_RESULTS[CATEGORY]['champion_ap']
    CHAMPION_AUC     = CATEGORY_RESULTS[CATEGORY]['champion_auc']
    MLFLOW_VERSION   = CATEGORY_RESULTS[CATEGORY]['mlflow_version']

    champion_model = mlflow.sklearn.load_model(f'runs:/{CHAMPION_RUN_ID}/model')
    joblib.dump(champion_model, f'/tmp/ps1_{tag}_champion.joblib')
    joblib.dump(FEATURE_COLS,   f'/tmp/ps1_{tag}_feature_cols.joblib')
    test_metrics = evaluate_all(champion_model, f'champion_{tag}_final')
    S3_CLIENT.put_object(Bucket=ARTIFACT_BUCKET,
                          Key=f'sagemaker/ps1-3d/{tag}/metrics/test_metrics.json',
                          Body=json.dumps(test_metrics), ContentType='application/json')
    with tarfile.open(f'/tmp/model_{tag}.tar.gz', 'w:gz') as tar:
        tar.add(f'/tmp/ps1_{tag}_champion.joblib',    arcname=f'ps1_{tag}_champion.joblib')
        tar.add(f'/tmp/ps1_{tag}_feature_cols.joblib', arcname=f'ps1_{tag}_feature_cols.joblib')
    MODEL_S3_KEY = f'sagemaker/ps1-3d/{tag}/model-v{MLFLOW_VERSION}/model.tar.gz'
    S3_CLIENT.upload_file(f'/tmp/model_{tag}.tar.gz', ARTIFACT_BUCKET, MODEL_S3_KEY)
    MODEL_DATA_URL = f's3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}'

    image_uri = sm_images.retrieve(framework='sklearn', region=REGION, version='1.2-1',
                                    instance_type='ml.m5.large', image_scope='inference')
    try:
        SM_CLIENT.create_model_package_group(
            ModelPackageGroupName=MODEL_REGISTRY_NAME,
            ModelPackageGroupDescription=f'PS1-3d {CATEGORY} — Chicago Ventra. will_fail_3d. v6.')
    except Exception: pass
    resp = SM_CLIENT.create_model_package(
        ModelPackageGroupName=MODEL_REGISTRY_NAME,
        ModelPackageDescription=f'{CHAMPION_NAME} | AP={CHAMPION_AP:.4f} | MLflow v{MLFLOW_VERSION}',
        InferenceSpecification={
            'Containers': [{'Image': image_uri, 'ModelDataUrl': MODEL_DATA_URL}],
            'SupportedContentTypes': ['text/csv', 'application/json'],
            'SupportedResponseMIMETypes': ['application/json'],
            'SupportedTransformInstanceTypes': ['ml.m5.large'],
            'SupportedRealtimeInferenceInstanceTypes': ['ml.m5.large'],
        },
        ModelApprovalStatus='PendingManualApproval',
        ModelMetrics={'ModelQuality': {'Statistics': {'ContentType': 'application/json',
            'S3Uri': f's3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/metrics/test_metrics.json'}}},
        CustomerMetadataProperties={
            'ps': 'PS1_3d', 'category': CATEGORY, 'target': 'will_fail_3d', 'city': 'chicago',
            'algorithm': CHAMPION_NAME, 'test_ap': str(round(CHAMPION_AP, 4)),
            'test_auc': str(round(CHAMPION_AUC, 4)),
            'mlflow_version': str(MLFLOW_VERSION), 'mlflow_run_id': CHAMPION_RUN_ID,
            'n_features': str(len(FEATURE_COLS))})
    SM_MODEL_PKG_ARN = resp['ModelPackageArn']
    CATEGORY_RESULTS[CATEGORY].update({
        'sm_pkg_arn': SM_MODEL_PKG_ARN, 'model_data_url': MODEL_DATA_URL,
        'champion_model': champion_model, 'X_test': cat['X_test'], 'y_test': cat['y_test'],
        'X_train': cat['X_train'],
    })
    print(f'  [{CATEGORY}] SM Package: ...{SM_MODEL_PKG_ARN[-40:]}')

print('\nAll categories registered in MLflow + SageMaker.')
for cn, res in CATEGORY_RESULTS.items():
    print(f"  {cn:<6}  champion={res['champion_name']:<30}  ",
          f"AUC={res['champion_auc']:.4f}  AP={res['champion_ap']:.4f}  thresh={res['threshold']:.3f}")


In [ ]:
# ── CELL 24 : Approve + deploy SageMaker Endpoints (per category) ────────────
# Uses boto3 directly (sagemaker SDK has unfixable circular imports mid-session).
# Injects inference.py into model.tar.gz so the sklearn container can load
# the joblib champion model without a custom image or ECR.
import boto3 as _b3_24, tarfile as _tf24, io as _io24
import pandas as _pd24

_s3c_24     = _b3_24.client('s3',                region_name=SM_CLIENT.meta.region_name)
_sm_runtime = _b3_24.client('sagemaker-runtime', region_name=SM_CLIENT.meta.region_name)

# sklearn container requires SAGEMAKER_PROGRAM env var + this file in the tar
_SKLEARN_IMAGE = '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3'
_INFERENCE_PY  = b'''import os, io, csv, glob
import numpy as np
import joblib

def model_fn(model_dir):
    hits = glob.glob(os.path.join(model_dir, '*_champion.joblib'))
    if not hits:
        raise FileNotFoundError(
            f'No *_champion.joblib in {model_dir}: {os.listdir(model_dir)}'
        )
    return joblib.load(hits[0])

def input_fn(request_body, content_type):
    if content_type == 'text/csv':
        return np.array(list(csv.reader(io.StringIO(request_body))), dtype=float)
    raise ValueError(f'Unsupported content type: {content_type}')

def predict_fn(input_data, model):
    return model.predict_proba(input_data)[:, 1]

def output_fn(prediction, accept):
    return ','.join(f'{p:.6f}' for p in prediction)
'''

PREDICTORS = {}

for CATEGORY, res in CATEGORY_RESULTS.items():
    cat      = CATEGORIES[CATEGORY]
    tag      = CATEGORY.lower()
    ep_name  = cat['endpoint']
    pkg_arn  = res['sm_pkg_arn']
    mdl_name = f'ps1-3d-{tag}-model'
    cfg_name = f'ps1-3d-{tag}-epcfg'

    # -- Approve model package
    SM_CLIENT.update_model_package(
        ModelPackageArn=pkg_arn, ModelApprovalStatus='Approved'
    )
    print(f'[{CATEGORY}] Approved: {pkg_arn}')

    # -- Get model S3 URI from model package, inject inference.py, upload patched tar
    _pkg_info   = SM_CLIENT.describe_model_package(ModelPackageName=pkg_arn)
    _orig_uri   = _pkg_info['InferenceSpecification']['Containers'][0]['ModelDataUrl']
    _orig_bucket, _orig_key = _orig_uri[5:].split('/', 1)   # strip s3://
    _new_key    = _orig_key.replace('/model-v1/', '/model-v1-handler/')

    _orig_bytes = _s3c_24.get_object(Bucket=_orig_bucket, Key=_orig_key)['Body'].read()
    _new_buf = _io24.BytesIO()
    with _tf24.open(fileobj=_io24.BytesIO(_orig_bytes), mode='r:gz') as _tin, \
         _tf24.open(fileobj=_new_buf, mode='w:gz') as _tout:
        for _m in _tin.getmembers():
            _tout.addfile(_m, _tin.extractfile(_m))
        _inf_info = _tf24.TarInfo(name='inference.py')
        _inf_info.size = len(_INFERENCE_PY)
        _tout.addfile(_inf_info, _io24.BytesIO(_INFERENCE_PY))
    _new_buf.seek(0)
    _s3c_24.put_object(Bucket=_orig_bucket, Key=_new_key, Body=_new_buf.read())
    _new_s3_uri = f's3://{_orig_bucket}/{_new_key}'
    print(f'[{CATEGORY}] Patched tar uploaded: {_new_s3_uri}')

    # -- Create SageMaker Model (direct container spec overrides model package image)
    try:
        SM_CLIENT.delete_model(ModelName=mdl_name)
    except Exception:
        pass
    SM_CLIENT.create_model(
        ModelName=mdl_name,
        ExecutionRoleArn=ROLE,
        PrimaryContainer={
            'Image':        _SKLEARN_IMAGE,
            'ModelDataUrl': _new_s3_uri,
            'Environment':  {
                'SAGEMAKER_PROGRAM':          'inference.py',
                'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml/model',
            },
        },
    )
    print(f'[{CATEGORY}] Model created: {mdl_name}')

    # -- Create endpoint config with data capture
    try:
        SM_CLIENT.delete_endpoint_config(EndpointConfigName=cfg_name)
    except Exception:
        pass
    SM_CLIENT.create_endpoint_config(
        EndpointConfigName=cfg_name,
        ProductionVariants=[{
            'VariantName':          'AllTraffic',
            'ModelName':            mdl_name,
            'InitialInstanceCount': 1,
            'InstanceType':         'ml.m5.large',
            'InitialVariantWeight': 1,
        }],
        DataCaptureConfig={
            'EnableCapture':             True,
            'InitialSamplingPercentage': 100,
            'DestinationS3Uri': (
                f's3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/data-capture/'
            ),
            'CaptureOptions': [
                {'CaptureMode': 'Input'},
                {'CaptureMode': 'Output'},
            ],
        },
    )
    print(f'[{CATEGORY}] Endpoint config created: {cfg_name}')

    # -- Create or update endpoint (handle Failed state: must delete before recreate)
    _existing = [e for e in SM_CLIENT.list_endpoints()['Endpoints']
                 if e['EndpointName'] == ep_name]
    if _existing:
        _ep_status = _existing[0]['EndpointStatus']
        if _ep_status == 'Failed':
            _fail_reason = SM_CLIENT.describe_endpoint(
                EndpointName=ep_name)['FailureReason']
            print(f'[{CATEGORY}] Endpoint was Failed: {_fail_reason}')
            print(f'[{CATEGORY}] Deleting failed endpoint ...')
            SM_CLIENT.delete_endpoint(EndpointName=ep_name)
            SM_CLIENT.get_waiter('endpoint_deleted').wait(
                EndpointName=ep_name,
                WaiterConfig={'Delay': 15, 'MaxAttempts': 40})
            SM_CLIENT.create_endpoint(
                EndpointName=ep_name, EndpointConfigName=cfg_name)
            print(f'[{CATEGORY}] Creating fresh endpoint: {ep_name}')
        else:
            SM_CLIENT.update_endpoint(
                EndpointName=ep_name, EndpointConfigName=cfg_name)
            print(f'[{CATEGORY}] Updating existing endpoint ({_ep_status}): {ep_name}')
    else:
        SM_CLIENT.create_endpoint(
            EndpointName=ep_name, EndpointConfigName=cfg_name)
        print(f'[{CATEGORY}] Creating endpoint: {ep_name}')

    # -- Wait for endpoint to be InService (up to 45 min)
    print(f'[{CATEGORY}] Waiting for endpoint (up to 45 min) ...')
    _waiter = SM_CLIENT.get_waiter('endpoint_in_service')
    try:
        _waiter.wait(EndpointName=ep_name,
                     WaiterConfig={'Delay': 30, 'MaxAttempts': 90})
        print(f'[{CATEGORY}] Endpoint live: {ep_name}')
        PREDICTORS[CATEGORY] = ep_name
    except Exception as _we:
        _ep_info = SM_CLIENT.describe_endpoint(EndpointName=ep_name)
        _status  = _ep_info['EndpointStatus']
        _reason  = _ep_info.get('FailureReason', '(none)')
        print(f'[{CATEGORY}] Waiter ended -- status={_status}  reason={_reason}')
        if _status == 'InService':
            print(f'[{CATEGORY}] Endpoint IS live (waiter false-alarm). Continuing.')
            PREDICTORS[CATEGORY] = ep_name
        elif _status in ('Creating', 'Updating'):
            print(f'[{CATEGORY}] Still {_status}. Re-run this cell once it finishes.')
        else:
            raise RuntimeError(
                f'{CATEGORY} endpoint in state={_status}: {_reason}'
            ) from _we

    # -- Smoke test: invoke with 3 test rows
    _X3   = _pd24.DataFrame(res['X_test'][:3], columns=FEATURE_COLS)
    _csv3 = _X3.to_csv(index=False, header=False)
    _resp = _sm_runtime.invoke_endpoint(
        EndpointName=ep_name,
        ContentType='text/csv',
        Body=_csv3,
    )
    _result = _resp['Body'].read().decode('utf-8')
    print(f'[{CATEGORY}] Sample predictions: {_result}')

print('\nAll endpoints deployed. PREDICTORS:', list(PREDICTORS.keys()))


In [ ]:
# ── CELL 25 : SageMaker Model Monitor — Data Quality Baselines (per category) ─
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.model_monitor import CronExpressionGenerator

MONITORS = {}
for CATEGORY, res in CATEGORY_RESULTS.items():
    tag = CATEGORY.lower()
    monitor = DefaultModelMonitor(role=ROLE, instance_count=1, instance_type="ml.m5.large",
                                   volume_size_in_gb=20, max_runtime_in_seconds=3600,
                                   sagemaker_session=SM_SESSION)
    baseline_df = pd.DataFrame(res["X_train"], columns=FEATURE_COLS)
    baseline_df.to_csv(f"/tmp/baseline_{tag}.csv", index=False)
    S3_CLIENT.upload_file(f"/tmp/baseline_{tag}.csv", ARTIFACT_BUCKET,
                          f"sagemaker/ps1-3d/{tag}/baseline/baseline.csv")
    monitor.suggest_baseline(
        baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/baseline/baseline.csv",
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/baseline/output/",
        wait=True, logs=True)
    print(f"[{CATEGORY}] Baseline complete.")
    MONITORS[CATEGORY] = monitor
print("All baselines done.")


In [ ]:
# ── CELL 26 : Schedule daily Data Quality monitoring (per category) ───────────
cw = boto3.client("cloudwatch", region_name=REGION)

for CATEGORY, monitor in MONITORS.items():
    tag = CATEGORY.lower()
    cat = CATEGORIES[CATEGORY]
    sched_name = f"chicago-ps1-3d-{tag}-data-quality"
    monitor.create_monitoring_schedule(
        monitor_schedule_name=sched_name, endpoint_input=cat["endpoint"],
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/monitor-output/",
        statistics=monitor.baseline_statistics(), constraints=monitor.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.daily(), enable_cloudwatch_metrics=True)
    print(f"[{CATEGORY}] Data Quality Monitor: {sched_name}")
    cw.put_metric_alarm(
        AlarmName=f"PS1-3d-{CATEGORY}-DataQuality-ViolationRate",
        ComparisonOperator="GreaterThanThreshold", EvaluationPeriods=1,
        MetricName="feature_baseline_drift_percentage",
        Namespace="aws/sagemaker/Endpoints/data-metrics",
        Period=86400, Statistic="Average", Threshold=3.0,
        AlarmDescription=f"PS1-3d {CATEGORY} input feature violation rate > 3%",
        Dimensions=[{"Name":"Endpoint","Value":cat["endpoint"]}],
        TreatMissingData="notBreaching")
    print(f"[{CATEGORY}] CW Alarm: PS1-3d-{CATEGORY}-DataQuality-ViolationRate")


In [ ]:
# ── CELL 27 : Model Quality Monitor (per category) ──────────────────────────
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

MQ_MONITORS = {}
for CATEGORY, res in CATEGORY_RESULTS.items():
    tag = CATEGORY.lower()
    cat = CATEGORIES[CATEGORY]
    mq = ModelQualityMonitor(role=ROLE, instance_count=1, instance_type="ml.m5.large",
                              sagemaker_session=SM_SESSION)
    y_prob_test = res["champion_model"].predict_proba(res["X_test"])[:,1]
    mq_df = pd.DataFrame({"prediction":y_prob_test, "probability":y_prob_test,
                           "label":res["y_test"].astype(str)})
    mq_df.to_csv(f"/tmp/mq_baseline_{tag}.csv", index=False)
    S3_CLIENT.upload_file(f"/tmp/mq_baseline_{tag}.csv", ARTIFACT_BUCKET,
                          f"sagemaker/ps1-3d/{tag}/mq-baseline/mq_baseline.csv")
    mq.suggest_baseline(
        baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-baseline/mq_baseline.csv",
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-baseline/output/",
        problem_type="BinaryClassification",
        inference_attribute="prediction", probability_attribute="probability",
        ground_truth_attribute="label", probability_threshold_attribute="0.5", wait=True)
    mq.create_monitoring_schedule(
        monitor_schedule_name=f"chicago-ps1-3d-{tag}-model-quality",
        endpoint_input=cat["endpoint"],
        ground_truth_input=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/ground-truth/",
        problem_type="BinaryClassification",
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-output/",
        constraints=mq.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.daily(), enable_cloudwatch_metrics=True)
    print(f"[{CATEGORY}] Model Quality Monitor: chicago-ps1-3d-{tag}-model-quality")
    print(f"  Ground truth: s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/ground-truth/")
    MQ_MONITORS[CATEGORY] = mq


In [ ]:
# ── CELL 28 : Final summary (per category) ────────────────────────────────────
print("="*65)
print("PS1-3d TVM+GATE -- Complete  (CUBIC MARS AWS v6)")
print("="*65)

for CATEGORY, res in CATEGORY_RESULTS.items():
    cat = CATEGORIES[CATEGORY]
    tag = CATEGORY.lower()
    print(f"\n-- {CATEGORY} --")
    print(f"  Champion       : {res['champion_name']}  AUC={res['champion_auc']:.4f}")
    print(f"  MLflow model   : {cat['model_name']}  v{res['mlflow_version']}  [Staging]")
    print(f"  SM Registry    : {cat['registry_name']}")
    print(f"  SM Endpoint    : {cat['endpoint']}")
    print(f"  Feature Store  : {cat['fs_group']}")
    print(f"  Data Monitor   : chicago-ps1-3d-{tag}-data-quality  (daily)")
    print(f"  Model Monitor  : chicago-ps1-3d-{tag}-model-quality  (daily)")
    print(f"  CW Alarm       : PS1-3d-{CATEGORY}-DataQuality-ViolationRate")

print(f"\n  Features       : {len(FEATURE_COLS)}")
print(f"  Artifacts      : s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/")
print("""
Retraining flow:
  1. Re-run from CELL 4 -- reads latest Gold, trains both TVM + GATE
  2. MLflow auto-increments version numbers per category
  3. SM Registry adds new packages (PendingManualApproval)
  4. Approve in Studio UI -> new Endpoints deployed
""")


In [ ]:
# ── FINAL CELL : Export notebook with all outputs to HTML ───────────────────
# Strategy:
#   1. Auto-discover notebook path on EFS
#   2. Force-save via Jupyter server REST API (flushes kernel outputs to disk)
#   3. Run nbconvert --to html
#   4. If outputs still missing, retry with --execute
import subprocess, os, datetime, time, re
try:
    import requests as _req
except ImportError:
    _req = None

_ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')

# -- Step 1: locate the notebook file
_nb_path = None
_search_names = ['PS1_3d_TVM-GATE_SageMaker_MLflow_FeatureStore.ipynb', 'PS1.ipynb']
for _name in _search_names:
    _r = subprocess.run(
        ['find', '/home/sagemaker-user', '-name', _name,
         '-not', '-path', '*/.ipynb_checkpoints/*'],
        capture_output=True, text=True, timeout=15
    )
    _hits = [p for p in _r.stdout.strip().splitlines() if os.path.exists(p)]
    if _hits:
        _nb_path = _hits[0]
        break
if not _nb_path:
    for _c in [os.path.join(os.getcwd(), _search_names[0]),
               '/home/sagemaker-user/PS1/PS1.ipynb']:
        if os.path.exists(_c):
            _nb_path = _c; break
if not _nb_path:
    print('[HTML] Notebook not found. Run: !find /home/sagemaker-user -name "*.ipynb" | grep -i ps1')
    raise SystemExit(1)
print(f'[HTML] Notebook: {_nb_path}')

# -- Step 2: force-save via Jupyter server API so outputs are in the file
_saved = False
if _req is not None:
    try:
        _srv = subprocess.run(['jupyter', 'server', 'list'],
                              capture_output=True, text=True, timeout=10)
        _m = re.search(r'http://[^\s]+', _srv.stdout + _srv.stderr)
        if _m:
            _srv_url = _m.group(0)
            _tok_m   = re.search(r'token=([a-f0-9]+)', _srv_url)
            _token   = _tok_m.group(1) if _tok_m else ''
            _base    = _srv_url.split('?')[0].rstrip('/')
            _rel     = _nb_path.replace('/home/sagemaker-user/', '', 1)
            _hdrs    = {'Authorization': f'token {_token}'}
            # POST /api/contents/<path>/checkpoints flushes current state to disk
            _sr = _req.post(f'{_base}/api/contents/{_rel}/checkpoints',
                            headers=_hdrs, timeout=15)
            if _sr.status_code in (200, 201):
                print(f'[HTML] Server save OK ({_sr.status_code}) — outputs flushed to disk')
                _saved = True
                time.sleep(2)
            # 403 is normal in SageMaker Studio (different auth layer) — silent fallback
    except Exception as _se:
        print(f'[HTML] Server save skipped: {_se}')
else:
    print('[HTML] requests not available — install with !pip install requests')

if not _saved:
    print('[HTML] Falling back to 5 s sleep for auto-save ...')
    time.sleep(5)

# -- Step 3: convert to HTML
_out_dir  = os.path.dirname(_nb_path)
_out_html = os.path.join(_out_dir, f'PS1_output_{_ts}.html')
_result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', _nb_path, '--output', _out_html],
    capture_output=True, text=True
)
if _result.returncode == 0 and os.path.exists(_out_html):
    _size = os.path.getsize(_out_html) / 1024
    print(f'[HTML] Saved: {_out_html}  ({_size:.0f} KB)')
    print('  Open in browser — all cell outputs preserved permanently.')
else:
    print(f'[HTML] nbconvert failed (rc={_result.returncode}):')
    if _result.stderr: print(_result.stderr[-600:])
    # -- Step 4: re-run everything and capture fresh outputs
    print('[HTML] Retrying with --execute (re-runs all cells, ~15-30 min) ...')
    _out_html2 = os.path.join(_out_dir, f'PS1_executed_{_ts}.html')
    _r2 = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', '--execute',
         '--allow-errors', '--ExecutePreprocessor.timeout=7200',
         _nb_path, '--output', _out_html2],
        capture_output=True, text=True
    )
    if _r2.returncode == 0 and os.path.exists(_out_html2):
        _size2 = os.path.getsize(_out_html2) / 1024
        print(f'[HTML] Executed export saved: {_out_html2}  ({_size2:.0f} KB)')
    else:
        print('[HTML] Both attempts failed.')
        print(_r2.stderr[-500:] if _r2.stderr else '(no stderr)')
